# MD meets Docking: An use case on antibody design

***
This tutorial aims to illustrate how we can use molecular dynamics simulations to improve a docking protocol for antibody-antigen complexes. We will use the following dataset of 11 antibody-antigen complexes from the [Protein-Protein Interaction Benchmark 5](https://www.sciencedirect.com/science/article/pii/S0022283615004180?via%3Dihub), which includes the PDB IDs of the bound and unbound structures, as well as the names of the proteins involved:
|           | Category |  PDB ID 1  | Protein 1                            | PDB ID 2	   | Protein 2                        	|  
|-----------|----------|------------|--------------------------------------|---------------|------------------------------------| 
|2VXT_HL:I	|     A	   | 2VXU_HL	| Murine reference antibody 125-2H Fab | 1J0S_A        | Interleukin-18	                    |
|2W9E_HL:A	|     A	   | 2W9D_HL    | ICSM 18 Fab fragment	               | 1QM1_A	       | Prion protein fragment             |
|3EOA_LH:I	|     A	   | 3EO9_LH    | Efalizumab Fab fragment              | 3F74_A	       | Integrin alpha-L I domain	        |
|3HMX_LH:AB	|     A	   | 3HMW_LH    | Ustekinumab Fab	                   | 1F45_AB       | Interleukin-12                 	|
|3MXW_LH:A	|     A	   | 3MXV_LH    | Anti-Shh 5E1 chimera Fab fragment	   | 3M1N_A	       | Sonic Hedgehog N-terminal domain   |
|5VPG_CD:A	|     A	   | 3RVT_CD    | 4C1 Fab	                           | 3F5V_A	       | DER P 1 allergen	                |
|4DN4_LH:M	|     A	   | 4DN3_LH    | CNTO888 Fab	                       | 1DOL_A	       | MCP-1                              |
|4FQI_HL:ABEFCD | A    | 4FQH_HL	| CR9114 Fab	                       | 2FK0_ABCDEF   | H5N1 influenza virus hemagglutinin	|
|4G6J_HL:A	|     A	   | 4G5Z_HL	| Canakinumab antibody fragment	       | 4I1B_A	       | Interleukin-1 beta	                |
|4G6M_HL:A	|     A	   | 4G6K_HL	| Gevokizumab antibody fragment	       | 4I1B_A	       | Interleukin-1 beta	                |
|4GXU_MN:ABEFCD	|A	   | 4GXV_HL	| 1F1 antibody	                       | 1RUZ_HIJKLM   | 1918H1 hemagglutinin	            |
***


The workflow is made of 3 sub-workflows:
1. HADDOCK3 docking
2. Antibody MD -> CDR loops clustering -> HADDOCK3 docking
3. HADDOCK3 docking  -> Ab-Ag AWH-MD -> CDR loops clustering -> HADDOCK3 docking 

In [1]:
import os
import re
import zipfile
import nglview as nv
import ipywidgets
import plotly.graph_objs as go
from pathlib import Path
from biobb_common import biobb_global_properties
from variables import *

biobb_global_properties.update({
    'out_log_path': 'log/log.out',  # Define the folder and name for
    'err_log_path': 'log/log.err',  # output and error logs
    'remove_tmp': True,
    'can_write_console_log': True,
    #'disable_sandbox': True,       # Disable the sandbox so input data is not copied
})

out_path = Path('data/bio2/')
RefAbAgs = (
    # Reference         Antibody   Antigen
    ("2VXT_HL:I",	   "2VXU_HL", "1J0S_A"),
    ("2W9E_HL:A",	   "2W9D_HL", "1QM1_A"),
    ("3EOA_LH:I",	   "3EO9_LH", "3F74_A"),
    ("3HMX_LH:AB",	   "3HMW_LH", "1F45_AB"),
    ("3MXW_LH:A",	   "3MXV_LH", "3M1N_A"),
    ("5VPG_CD:A",	   "3RVT_CD", "3F5V_A"),
    ("4DN4_LH:M",	   "4DN3_LH", "1DOL_A"),
    ("4FQI_HL:ABEFCD", "4FQH_HL", "2FK0_ABCDEF"),
    ("4G6J_HL:A",      "4G5Z_HL", "4I1B_A"),
    ("4G6M_HL:A" ,     "4G6K_HL", "4I1B_A"),        # This is the same than in the HADDOCK workflow
    ("4GXU_MN:ABEFCD", "4GXV_HL", "1RUZ_HIJKLM"),
)

In [2]:
def pdb_tools_pipeline(inp_file, out_file, steps):
    """Helper function to concatenate calls to pdb_tools"""
    tmp_file = inp_file
    for step, props in steps:
        # Apply each step in the pipeline
        step(input_file_path  = tmp_file, 
             output_file_path = out_file, 
             properties       = props)
        tmp_file = 'tmp.pdb'
        os.rename(out_file, tmp_file)
    os.rename(tmp_file,out_file)

## HADDOCK3 baseline

<a id="fetch"></a>
***
### Fetching PDB structures
Downloading **PDB structures** with the **protein molecule** from the RCSB PDB database.<br>
***
**Building Blocks** used:
 - [Pdb](https://biobb-io.readthedocs.io/en/latest/api.html#module-api.pdb) from **biobb_io.api.pdb**
***

In [ ]:
# Downloading desired PDB file 
# Import module
from biobb_io.api.pdb import pdb

prep_dir = out_path / '0_base'
prep_dir.mkdir(parents=True, exist_ok=True)
flat_pdb_list = [pdb for trio in RefAbAgs for pdb in trio]

for pdb_chs in flat_pdb_list:
    # Create properties dict and inputs/outputs
    downloaded_pdb = prep_dir / f'{pdb_chs}.pdb'
    pdb_code = pdb_chs.split('_')[0]
    prop = {'pdb_code': pdb_code}

    # Create and launch bb
    pdb(output_pdb_path=downloaded_pdb,
        properties=prop)

2026-05-05 16:19:17,229 [MainThread  ] [INFO ]  Module: biobb_io.api.pdb Version: 5.2.3
2026-05-05 16:19:17,229 [MainThread  ] [INFO ]  Downloading 2vxt from: https://www.ebi.ac.uk/pdbe/entry-files/download/pdb2vxt.ent
2026-05-05 16:19:17,687 [MainThread  ] [INFO ]  Writting pdb to: data/bio2/0_base/2VXT_HL:I.pdb
2026-05-05 16:19:17,688 [MainThread  ] [INFO ]  Filtering lines NOT starting with one of these words: ['ATOM', 'MODEL', 'ENDMDL']
2026-05-05 16:19:17,691 [MainThread  ] [INFO ]  
2026-05-05 16:19:17,693 [MainThread  ] [INFO ]  Module: biobb_io.api.pdb Version: 5.2.3
2026-05-05 16:19:17,694 [MainThread  ] [INFO ]  Downloading 2vxu from: https://www.ebi.ac.uk/pdbe/entry-files/download/pdb2vxu.ent
2026-05-05 16:19:18,087 [MainThread  ] [INFO ]  Writting pdb to: data/bio2/0_base/2VXU_HL.pdb
2026-05-05 16:19:18,088 [MainThread  ] [INFO ]  Filtering lines NOT starting with one of these words: ['ATOM', 'MODEL', 'ENDMDL']
2026-05-05 16:19:18,091 [MainThread  ] [INFO ]  
2026-05-05 16:

In [33]:
pdb_files = [out_path / '0_base' / f'{pdb_chs}.pdb' for pdb_chs in RefAbAgs[9]]
view1 = nv.show_structure_file(str(pdb_files[0]))
view2 = nv.show_structure_file(str(pdb_files[1]))
view3 = nv.show_structure_file(str(pdb_files[2]))

views = [view1, view2, view3]
for view in views:
    view._remote_call('setSize', target='Widget', args=['500px','500px'])
    view.layout.margin = "auto"
    view.camera='orthographic'
box = ipywidgets.HBox(views)
box   

### Preparing the structures for docking

[pdb format](https://www.bonvinlab.org/haddock3-user-manual/structure_requirements.html#pdb-format)

In [9]:
ab_pdb_src = str(out_path / '0_base' / f'{ab}.pdb')

In [10]:
from anarcii import Anarcii
model = Anarcii(verbose=True)
model.number(ab_pdb_src)

Using device CPU with 16 CPUs
Batch size: 32
	Speed is a balance of batch size and length diversity. Adjust accordingly. For a full explanation see:
 	wiki/FAQs#recommended-batch-sizes
 	Seqs all similar length (+/-5), increase batch size. Mixed lengths (+/-30), reduce.


Consider larger batch size for optimal GPU performance.

Length of sequence list: 2
Processing sequences in 1 chunks of 102400 sequences.
Processing chunk 1 of 1.

 2 Long sequences detected - running in sliding window. This is slow.
Max probability windows selected.

Making predictions on 1 batches.
PDBx model index, chain ID: 0, H
ANARCII chain type (score): H (30.586580276489258)
 Sequence length: 128
 Sequence: [((1, ' '), 'Q'), ((2, ' '), 'V'), ((3, ' '), 'Q'), ((4, ' '), 'L'), ((5, ' '), 'Q'), ((6, ' '), 'E'), ((7, ' '), 'S'), ((8, ' '), 'G'), ((9, ' '), 'P'), ((10, ' '), '-'), ((11, ' '), 'G'), ((12, ' '), 'L'), ((13, ' '), 'V'), ((14, ' '), 'K'), ((15, ' '), 'P'), ((16, ' '), 'S'), ((17, ' '), 'Q'), ((18, ' ')

{(0,
  'H'): {'numbering': [((1, ' '), 'Q'),
   ((2, ' '), 'V'),
   ((3, ' '), 'Q'),
   ((4, ' '), 'L'),
   ((5, ' '), 'Q'),
   ((6, ' '), 'E'),
   ((7, ' '), 'S'),
   ((8, ' '), 'G'),
   ((9, ' '), 'P'),
   ((10, ' '), '-'),
   ((11, ' '), 'G'),
   ((12, ' '), 'L'),
   ((13, ' '), 'V'),
   ((14, ' '), 'K'),
   ((15, ' '), 'P'),
   ((16, ' '), 'S'),
   ((17, ' '), 'Q'),
   ((18, ' '), 'T'),
   ((19, ' '), 'L'),
   ((20, ' '), 'S'),
   ((21, ' '), 'L'),
   ((22, ' '), 'T'),
   ((23, ' '), 'C'),
   ((24, ' '), 'S'),
   ((25, ' '), 'F'),
   ((26, ' '), 'S'),
   ((27, ' '), 'G'),
   ((28, ' '), 'F'),
   ((29, ' '), 'S'),
   ((30, ' '), 'L'),
   ((31, ' '), 'S'),
   ((32, ' '), '-'),
   ((33, ' '), '-'),
   ((34, ' '), 'T'),
   ((35, ' '), 'S'),
   ((36, ' '), 'G'),
   ((37, ' '), 'M'),
   ((38, ' '), 'G'),
   ((39, ' '), 'V'),
   ((40, ' '), 'G'),
   ((41, ' '), 'W'),
   ((42, ' '), 'I'),
   ((43, ' '), 'R'),
   ((44, ' '), 'Q'),
   ((45, ' '), 'P'),
   ((46, ' '), 'S'),
   ((47, ' '), 'G'

In [11]:
from biobb_pdb_tools.pdb_tools import  *
from biobb_pdb_tools.pdb_tools.biobb_pdb_merge import biobb_pdb_merge
from anarcii import Anarcii


prep_dir = out_path / '1_pre'
prep_dir.mkdir(parents=True, exist_ok=True)

model = Anarcii()
# Mapping of original residue numbers to new residue numbers for each chain
resmap = {}
def prepare_antibody(ab_pdb_src, ab_pdb_out):
    ab_pdb_chains = []
    rgx = r'.*/([\w]{4})_([\w]+):?\w*?\.pdb$'
    match = re.match(rgx, ab_pdb_src)
    ab_pdb_code, chs = match.groups()
    results = model.number(ab_pdb_src)
    resmap[ab] = {}
    for ch in chs:
        pdb_chain = str(out_path / '1_pre' / f'{ab_pdb_code}_V{ch}.pdb')
        ab_pdb_chains.append(pdb_chain)
        last_resnum = results[(0,ch)]['query_end']+1
        resmap[ab][ch] = last_resnum
        # The selection is always from 1, the start of the Fv
        sel = '1:'+str(last_resnum)
        steps = [
            (biobb_pdb_tidy.biobb_pdb_tidy,           {'strict': True}),      # Adhere to the format specifications
            (biobb_pdb_selchain.biobb_pdb_selchain,   {'chains': ch}),        # Extract chain
            (biobb_pdb_delhetatm.biobb_pdb_delhetatm, {}),                    # Remove all HETATM records 
            (biobb_pdb_fixinsert.biobb_pdb_fixinsert, {}),                    # Delete insertion codes and shift residue numbering 
            (biobb_pdb_selaltloc.biobb_pdb_selaltloc, {}),                    # Select altloc labels (highest occupancy) 
            (biobb_pdb_keepcoord.biobb_pdb_keepcoord, {}),                    # Remove all non-coordinate records 
            (biobb_pdb_selres.biobb_pdb_selres,       {'selection': sel }),   # Select residues by their range
            (biobb_pdb_tidy.biobb_pdb_tidy,           {})                     # Adhere to the format specifications
        ]
        # Create and launch bb
        pdb_tools_pipeline(ab_pdb_src, pdb_chain, steps)
    # Join the generated PDB files in a single ZIP file 
    zip_file_path =str(prep_dir / f'{ab}_Fv.zip')
    # Create a zip file and add the pdb_out file to it
    with zipfile.ZipFile(zip_file_path, 'w') as zipf:
        for pdb_chain in ab_pdb_chains:
            zipf.write(pdb_chain, arcname=pdb_chain)
    # Create and launch bb
    ab_pdb_merged = str(prep_dir / f'{ab}_Fv.pdb')
    biobb_pdb_merge(
        input_file_path  = zip_file_path,
        output_file_path = ab_pdb_merged,
    )
    steps = [
        (biobb_pdb_reres.biobb_pdb_reres,         {'number': 1}),     # Renumber the residues starting from 1
        (biobb_pdb_chain.biobb_pdb_chain,         {'chain': 'A'}),    # Modify the chain identifier column 
        (biobb_pdb_chainxseg.biobb_pdb_chainxseg, {}),                # Swap the segment identifier for the chain identifier
        (biobb_pdb_tidy.biobb_pdb_tidy,           {'strict': True})   # Adhere to the format specifications
    ]
    # Create and launch bb
    pdb_tools_pipeline(ab_pdb_merged, ab_pdb_out, steps)

def prepare_antigen(ag_pdb_src, ag_pdb_out):
    rgx = r'.*_[\w]*:?([\w]+)\.pdb$'
    match = re.match(rgx, ag_pdb_src)
    chs = match.groups()[0]
    chains = ','.join(list(chs.split(',')))
    # Create and launch bb
    steps = [
        (biobb_pdb_tidy.biobb_pdb_tidy,           {'strict': True}),   # Adhere to the format specifications
        (biobb_pdb_selchain.biobb_pdb_selchain,   {'chains': chains}),  # Extract chains
        (biobb_pdb_chain.biobb_pdb_chain,         {'chain': 'B'}),     # Modify the chain identifier column 
        (biobb_pdb_chainxseg.biobb_pdb_chainxseg, {}),                 # Swap the segment identifier for the chain identifier
        (biobb_pdb_delhetatm.biobb_pdb_delhetatm, {}),                 # Remove all HETATM records 
        (biobb_pdb_fixinsert.biobb_pdb_fixinsert, {}),                  # Delete insertion codes and shift residue numbering
        (biobb_pdb_selaltloc.biobb_pdb_selaltloc, {}),                 # Select altloc labels (highest occupancy)
        (biobb_pdb_keepcoord.biobb_pdb_keepcoord, {}),                 # Remove all non-coordinate records
        (biobb_pdb_tidy.biobb_pdb_tidy,           {'strict': True})    # Adhere to the format specifications
    ]
    pdb_tools_pipeline(ag_pdb_src, ag_pdb_out, steps)


for ref, ab, ag in RefAbAgs[9:10]:
    # PREAPRE THE ANTIBODY
    ab_pdb_src = str(out_path / '0_base' / f'{ab}.pdb')
    ab_pdb_clean = str(prep_dir / f'{ab}_clean.pdb')
    prepare_antibody(ab_pdb_src, ab_pdb_clean)
    # PREPARE THE ANTIGEN
    ag_pdb_src = str(out_path / '0_base' / f'{ag}.pdb')
    ag_pdb_clean = str(prep_dir / f'{ag}_clean.pdb')
    prepare_antigen(ag_pdb_src, ag_pdb_clean)
    # PREPARE THE REFERENCE COMPLEX
    ref_pdb_src = str(out_path / '0_base' / f'{ref}.pdb')
    ref_pdb_ab  = str(prep_dir / f'{ref}_ab.pdb')
    prepare_antibody(ref_pdb_src, ref_pdb_ab)
    ref_pdb_ag = str(prep_dir / f'{ref}_ag.pdb')
    prepare_antigen(ref_pdb_src, ref_pdb_ag)
    ## Join the generated PDB files in a single ZIP file ##
    zip_file_path = str(prep_dir / f'{ref}.zip')
    # Create a zip file and add the pdb_out file to it
    with zipfile.ZipFile(zip_file_path, 'w') as zipf:
        zipf.write(ref_pdb_ab, arcname=f'{ref}_antibody.pdb')
        zipf.write(ref_pdb_ag, arcname=f'{ref}_antigen.pdb')
    # Create properties dict and inputs/outputs
    ref_pdb_clean = str(prep_dir / f'{ref}_clean.pdb')
    steps = [
        (biobb_pdb_merge,         {}),                                 # Merges several PDB files into one
        (biobb_pdb_tidy.biobb_pdb_tidy,           {'strict': True})    # Adhere to the format specifications
    ]
    # Create and launch bb
    pdb_tools_pipeline(zip_file_path, ref_pdb_clean, steps)
    

2026-05-20 11:54:01,011 [MainThread  ] [INFO ]  Module: biobb_pdb_tools.pdb_tools.biobb_pdb_tidy Version: 5.2.1
2026-05-20 11:54:01,013 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_793f5e9e-b4b7-4a2b-88cb-8d0c22bc9b51
2026-05-20 11:54:01,014 [MainThread  ] [INFO ]  Copy to stage: data/bio2/0_base/4G6K_HL.pdb --> sandbox_793f5e9e-b4b7-4a2b-88cb-8d0c22bc9b51
2026-05-20 11:54:01,015 [MainThread  ] [INFO ]  Appending optional boolean property
2026-05-20 11:54:01,016 [MainThread  ] [INFO ]  pdb_tidy -strict data/bio2/0_base/4G6K_HL.pdb > data/bio2/1_pre/4G6K_VH.pdb
2026-05-20 11:54:01,016 [MainThread  ] [INFO ]  Creating command line with instructions and required arguments
2026-05-20 11:54:01,017 [MainThread  ] [INFO ]  Launching command (it may take a while): pdb_tidy -strict data/bio2/0_base/4G6K_HL.pdb > data/bio2/1_pre/4G6K_VH.pdb
2026-05-20 11:54:01,041 [MainThread  ] [INFO ]  Command 'pdb_tidy -

### Preparing the restrains

In [67]:
ref, ab, ag = RefAbAgs[9]

### Generate AIRs with predicted epitope (old)

In [ ]:
!arctic3d-restraints --r1 {arc_path} --r2 {arc_path} --run_dir {dock_dir}arctic3d_res/

[2025-06-04 17:40:04,713 cli_restraints INFO] Starting arctic3d-restraints
[2025-06-04 17:40:04,740 output INFO] Creating output_directory data/antibody/2_docking/arctic3d_res
[2025-06-04 17:40:04,740] cli_restraints.py:arctic3d:main:260: Input files are {'r1_res_fname': PosixPath('data/antibody/2_docking/arctic3d/clustered_residues_probs.out'), 'r2_res_fname': PosixPath('data/antibody/2_docking/arctic3d/clustered_residues_probs.out')}
[2025-06-04 17:40:04,740] output.py:arctic3d.log:setup_output_folder:64: Setting up output folder data/antibody/2_docking/arctic3d_res
[2025-06-04 17:40:04,740] output.py:arctic3d.log:setup_output_folder:73: File clustered_residues_probs.out already exists, adding _1 to the filename
[2025-06-04 17:40:04,740] cli_restraints.py:arctic3d:main:268: data/antibody/2_docking/arctic3d/ residues = {1: [120, 121, 122, 127, 129, 130, 131, 135, 136, 137, 138, 139, 140, 141, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 162, 164, 167, 168, 169, 170, 172

In [ ]:
# Obtain epitope
arc_path = 'data/antibody/2_docking/arctic3d/'
!arctic3d P01584 --db /home/rchaves/repo/arctic3d/db/swissprot --pdb_to_use 4I1B --run_dir {arc_path}

[2025-06-04 17:51:09,480 cli INFO] 
##############################################
#                                            #
#                 ARCTIC-3D                  #
#                                            #
##############################################

Starting ARCTIC-3D 0.4.1


Traceback (most recent call last):
  File "/home/rchaves/miniforge3/envs/biobb_wf_antibody/bin/arctic3d", line 8, in <module>
    sys.exit(maincli())
  File "/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.10/site-packages/arctic3d/cli.py", line 169, in maincli
    return cli(argument_parser, main)
  File "/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.10/site-packages/arctic3d/cli.py", line 164, in cli
    return main_func(**vars(cmd))
  File "/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.10/site-packages/arctic3d/cli.py", line 213, in main
    run_dir_path = create_output_folder(run_dir, uniprot_id)
  File "/home/rchaves/miniforge3/envs/biobb_wf_antibo

In [ ]:
epitopes = !cat {arc_path}clustered_residues.out

# Convert space-separated string to list of integers
residue_numbers1 = list(map(int, epitopes[0].split('>')[1].split()))

[119,
 120,
 121,
 122,
 125,
 127,
 129,
 130,
 131,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 143,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 157,
 162,
 164,
 167,
 168,
 169,
 170,
 171,
 172,
 180,
 181,
 182,
 183,
 188,
 189,
 190,
 191,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 208,
 209,
 210,
 212,
 213,
 214,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 231,
 232,
 233,
 234,
 242,
 243,
 244,
 245,
 246,
 247,
 256,
 257,
 263,
 264,
 265,
 266,
 267,
 268,
 269]

In [ ]:
# Convert to Uniprot numbering
!pdb_fetch 4I1B | pdb_reres -119 > {dock_dir}ag_uni.pdb

In [ ]:
view = nv.show_structure_file(f'{dock_dir}ag_uni.pdb')
for cluster, color in zip(epitopes, ['red', 'blue', 'green', 'orange', 'purple']):
    residue_numbers = list(map(int, cluster.split('>')[1].split()))
    res_sel = f"{', '.join(map(str, residue_numbers))}"
    print(res_sel)
    view.add_ball_and_stick(selection=res_sel, color=color, radius=0.2)
#view.add_surface(selection='not HOH',color='residueindex')
view

119, 120, 121, 122, 125, 127, 129, 130, 131, 135, 136, 137, 138, 139, 140, 141, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 157, 162, 164, 167, 168, 169, 170, 171, 172, 180, 181, 182, 183, 188, 189, 190, 191, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 208, 209, 210, 212, 213, 214, 219, 220, 221, 222, 223, 224, 225, 231, 232, 233, 234, 242, 243, 244, 245, 246, 247, 256, 257, 263, 264, 265, 266, 267, 268, 269
119, 120, 121, 122, 123, 159, 177, 178, 179, 182, 183, 184, 203, 204, 206, 207, 269
130, 170, 220, 222, 223, 225, 227, 242, 253, 254, 255, 256, 257, 258, 260, 261


NGLWidget()

In [ ]:
residue_numbers

[119,
 120,
 121,
 122,
 125,
 127,
 129,
 130,
 131,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 143,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 157,
 162,
 164,
 167,
 168,
 169,
 170,
 171,
 172,
 180,
 181,
 182,
 183,
 188,
 189,
 190,
 191,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 208,
 209,
 210,
 212,
 213,
 214,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 231,
 232,
 233,
 234,
 242,
 243,
 244,
 245,
 246,
 247,
 256,
 257,
 263,
 264,
 265,
 266,
 267,
 268,
 269]

In [ ]:
view = nv.show_structure_file(f'{dock_dir}ref_clean.pdb')
# view.clear()
residue_numbers = list(map(int, cluster.split('>')[1].split()))
res_sel = f"{', '.join(map(str, residue_numbers))}"
print(res_sel)
view.add_ball_and_stick(selection=res_sel, color=color, radius=0.2)
view

NGLWidget()

### Get the contact residues in the interface

In [13]:
!haddock-restraints interface {ref_pdb_clean} 3.9

Chain A: [32, 54, 55, 56, 58, 59, 60, 102, 103, 104, 147, 152, 170, 173, 211, 212, 213, 214]
Chain B: [72, 73, 74, 75, 81, 83, 84, 86, 89, 90, 92, 94, 96, 97, 98, 115, 116, 117, 118]


In [14]:
import MDAnalysis as mda
paratope = [32, 54, 55, 56, 58, 59, 60, 102, 103, 104, 147, 152, 170, 173, 211, 212, 213, 214]
epitope  = [72, 73, 74, 75, 81, 83, 84, 86, 89, 90, 92, 94, 96, 97, 98, 115, 116, 117, 118]
u = mda.Universe(ref_pdb_clean)
view = nv.show_mdanalysis(u)
view.add_trajectory(u)
view.clear_representations()
view.add_licorice(selection=f"( {', '.join(map(str, paratope))} ) and :A", color='red', name='paratope')
view.add_licorice(selection=f"( {', '.join(map(str, epitope))} ) and :B", color='blue', name='epitope')
view

NGLWidget()

### Get pasive from active

In [93]:
import json
# https://www.bonvinlab.org/haddock-restraints/configuration_file.html
tbl_config =[
  {
    "id": 1,
    "chain": "A",
    "active": [32, 54, 55, 56, 58, 59, 60, 102, 103, 104, 147, 152, 170, 173, 211, 212, 213, 214],
    "passive": [],
    "target": [2]
  },
  {
    "id": 2,
    "chain": "B",
    "active": [72, 73, 74, 75, 81, 83, 84, 86, 89, 90, 92, 94, 96, 97, 98, 115, 116, 117, 118],
    "passive": [],
    "target": [1]
  }
]
tbl_file_path = out_path / '1_pre' / 'tbl_config.json'
with open(tbl_file_path, "w") as f:
    json.dump(tbl_config, f, indent=2)

In [94]:
# haddock-restraints tbl path/to/config.json > restraints.tbl
!haddock-restraints tbl {tbl_file_path} > {out_path / '1_pre' / 'ambig-restraints.tbl'}

In [95]:
!head {out_path / '1_pre' / 'ambig-restraints.tbl'}

assign ( resid 32 and segid A )
       (
        ( resid 72 and segid B )
     or
        ( resid 73 and segid B )
     or
        ( resid 74 and segid B )
     or
        ( resid 75 and segid B )
     or


In [ ]:
# Tie antibody chains together
from biobb_haddock.haddock_restraints.haddock3_restrain_bodies import haddock3_restrain_bodies

# Create output
body_tbl = f'{out_path}/1_pre/unambig-restraints.tbl'

# Create and launch bb
haddock3_restrain_bodies( 
    input_structure_path=str(out_path / '1_pre' / f'{ref}_clean.pdb'),
    output_tbl_path=body_tbl
)

2026-05-12 12:47:59,172 [MainThread  ] [INFO ]  Module: biobb_haddock.haddock_restraints.haddock3_restrain_bodies Version: 5.2.1
2026-05-12 12:47:59,173 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a9141441-784d-4651-8e6b-9a15c3cd8110
2026-05-12 12:47:59,173 [MainThread  ] [INFO ]  Copy to stage: data/bio2/1_pre/4G6M_HL:A_clean.pdb --> sandbox_a9141441-784d-4651-8e6b-9a15c3cd8110
2026-05-12 12:47:59,173 [MainThread  ] [INFO ]  Launching command (it may take a while): haddock3-restraints restrain_bodies /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a9141441-784d-4651-8e6b-9a15c3cd8110/4G6M_HL:A_clean.pdb > /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a9141441-784d-4651-8e6b-9a15c3cd8110/unambig-restraints.tbl 2>&1
2026-05-12 12:47:59,540 [MainThread  ] [INFO ]  Command 'haddock3-restraints restrain_bodies /home/rchaves/repo/biobb/bi

0

### Running the docking

In [4]:
ref, ab, ag = RefAbAgs[9]
# https://www.bonvinlab.org/haddock3-user-manual/docking_scenarios/antibody-antigen.html
# https://github.com/haddocking/haddock3/blob/main/examples/docking-antibody-antigen/docking-antibody-antigen-CDR-NMR-CSP-full.cfg
config = f"""
# ====================================================================
# Protein-protein docking example with NMR-derived ambiguous interaction restraints

# directory in which the scoring will be done
run_dir = "{out_path / '2_dock' / 'run'}"

# execution mode
mode = "local"
ncores = 12
debug = true

# molecules to be docked
molecules =  [
    "{out_path / '1_pre' / f'{ab}_clean.pdb'}",
    "{out_path / '1_pre' / f'{ag}_clean.pdb'}"
    ]

# ====================================================================
# Parameters for each stage are defined below, prefer full paths
# ====================================================================
[topoaa]

[rigidbody]
tolerance = 5
# CDR to surface ambig restraints
ambig_fname = "{out_path / '1_pre' / 'ambig-restraints.tbl'}"
# Restraints to keep the antibody chains together
unambig_fname = "{out_path / '1_pre' / 'unambig-restraints.tbl'}"

[caprieval]
reference_fname = "{out_path / '1_pre' / f'{ref}_clean.pdb'}"

[seletop]
select = 200

[caprieval]
reference_fname = "{out_path / '1_pre' / f'{ref}_clean.pdb'}"

[flexref]
tolerance = 5
# CDR to surface ambig restraints
ambig_fname = "{out_path / '1_pre' / 'ambig-restraints.tbl'}"
# Restraints to keep the antibody chains together
unambig_fname = "{out_path / '1_pre' / 'unambig-restraints.tbl'}"

[caprieval]
reference_fname = "{out_path / '1_pre' / f'{ref}_clean.pdb'}"

[emref]
tolerance = 5
# CDR to surface ambig restraints
ambig_fname = "{out_path / '1_pre' / 'ambig-restraints.tbl'}"
# Restraints to keep the antibody chains together
unambig_fname = "{out_path / '1_pre' / 'unambig-restraints.tbl'}"

[caprieval]
reference_fname = "{out_path / '1_pre' / f'{ref}_clean.pdb'}"

[clustfcc]

[seletopclusts]
top_models = 4

[caprieval]
reference_fname = "{out_path / '1_pre' / f'{ref}_clean.pdb'}"

# ====================================================================
"""

dock_dir = out_path / '2_dock'
dock_dir.mkdir(parents=True, exist_ok=True)

with open(dock_dir / 'haddock_config.cfg', "w") as f:
    f.write(config)

In [102]:
print(config)


# ====================================================================
# Protein-protein docking example with NMR-derived ambiguous interaction restraints

# directory in which the scoring will be done
run_dir = "data/bio2/2_dock/run"

# execution mode
mode = "local"
ncores = 12
debug = true

# molecules to be docked
molecules =  [
    "data/bio2/1_pre/4G6K_HL_clean.pdb",
    "data/bio2/1_pre/4I1B_A_clean.pdb"
    ]

# ====================================================================
# Parameters for each stage are defined below, prefer full paths
# ====================================================================
[topoaa]

[rigidbody]
tolerance = 5
# CDR to surface ambig restraints
ambig_fname = "data/bio2/1_pre/ambig-restraints.tbl"
# Restraints to keep the antibody chains together
unambig_fname = "data/bio2/1_pre/unambig-restraints.tbl"

[caprieval]
reference_fname = "data/bio2/1_pre/4G6M_HL:A_clean.pdb"

[seletop]
select = 200

[caprieval]
reference_fname = "data/bio2/1_pre/

In [103]:
!rm -r {out_path / '2_dock' / 'run'} 2>/dev/null || true
!haddock3 {out_path / '2_dock'/ 'haddock_config.cfg'} 

[2026-05-12 12:55:42,751 cli INFO] 
##############################################
#                                            #
#                 HADDOCK3                   #
#                                            #
##############################################

!! Some of the HADDOCK3 components use CNS (Crystallographic and NMR System) which is free of use for non-profit applications. !!
!! For commercial use it is your own responsibility to have a proper license. !!
!! For details refer to the DISCLAIMER file in the HADDOCK3 repository. !!

Starting HADDOCK3 v2025.11.0 on 2026-05-12 12:55:00

Python 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]

[2026-05-12 12:55:46,190 libworkflow INFO] Reading instructions step 0_topoaa
[2026-05-12 12:55:46,190 libworkflow INFO] Reading instructions step 1_rigidbody
[2026-05-12 12:55:46,190 libworkflow INFO] Reading instructions step 2_caprieval
[2026-05-12 12:55:46,190 libworkflow INFO] Reading instructio

## MD generated CDR clusters

In [102]:
# https://mmb.irbbarcelona.org/biobb/workflows/tutorials/biobb_wf_flexdyn ?
# https://github.com/bioexcel/biobb_wf_flexdyn
# https://github.com/alevil-gmx/workflow_template/blob/main/workflow_template/workshop-template-ab.ipynb

In [4]:
MD_dir = out_path / '3_MD'
MD_dir.mkdir(parents=True, exist_ok=True)

<a id="fix"></a>
***
### Fix protein structure
**Checking** and **fixing** (if needed) the protein structure:<br>
- **Modeling** **missing side-chain atoms**, modifying incorrect **amide assignments**, choosing **alternative locations**.<br>
- **Checking** for missing **backbone atoms**, **heteroatoms**, **modified residues** and possible **atomic clashes**.

***
**Building Blocks** used:
 - [FixSideChain](https://biobb-model.readthedocs.io/en/latest/model.html#module-model.fix_side_chain) from **biobb_model.model.fix_side_chain**
***

In [25]:
# Check & Fix PDB
# Import module
from biobb_model.model.fix_side_chain import fix_side_chain

# Create prop dict and inputs/outputs
ab_pdb_src = str(out_path / '0_base' / f'{ab}.pdb')
fixed_pdb = str(MD_dir / f'{ab}_clean.pdb')

# Create and launch bb
fix_side_chain(input_pdb_path=ab_pdb_src, 
               output_pdb_path=fixed_pdb)

2026-05-14 12:22:43,828 [MainThread  ] [INFO ]  Module: biobb_model.model.fix_side_chain Version: 5.2.1
2026-05-14 12:22:43,829 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c034d1f8-a6d0-462b-a194-35e38b62381f
2026-05-14 12:22:43,830 [MainThread  ] [INFO ]  Copy to stage: data/bio2/0_base/4G6K_HL.pdb --> sandbox_c034d1f8-a6d0-462b-a194-35e38b62381f
2026-05-14 12:22:43,830 [MainThread  ] [INFO ]  Launching command (it may take a while): check_structure -i /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c034d1f8-a6d0-462b-a194-35e38b62381f/4G6K_HL.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c034d1f8-a6d0-462b-a194-35e38b62381f/4G6K_HL_clean.pdb --force_save fixside --fix ALL
2026-05-14 12:22:44,081 [MainThread  ] [INFO ]  Command 'check_structure -i /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/...' finalized wit

0

<a id="top"></a>
***
### Create protein system topology
**Building GROMACS topology** corresponding to the protein structure.<br>
Force field used in this tutorial is **charmm36** <br>

Generating two output files: 
- **GROMACS structure** (gro file)
- **GROMACS topology** ZIP compressed file containing:
    - *GROMACS topology top file* (top file)
    - *GROMACS position restraint file/s* (itp file/s)
***
**Building Blocks** used:
 - [Pdb2gmx](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.pdb2gmx) from **biobb_gromacs.gromacs.pdb2gmx**
***

In [131]:
# Downloading CHARMM36 FF
!curl  -o {MD_dir}/charmm36-jul2017.ff.tgz https://mackerell.umaryland.edu/download.php?filename=CHARMM_ff_params_files/charmm36-jul2017.ff.tgz
# Unzip CHARMM36 FF
!tar -xvzf {MD_dir}/charmm36-jul2017.ff.tgz -C {MD_dir}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  586k  100  586k    0     0   345k      0  0:00:01  0:00:01 --:--:--  345k
charmm36-jul2017.ff/
charmm36-jul2017.ff/spc.itp
charmm36-jul2017.ff/ffbonded.itp
charmm36-jul2017.ff/merged.rtp
charmm36-jul2017.ff/nbfix.itp
charmm36-jul2017.ff/gb.itp
charmm36-jul2017.ff/forcefield.doc
charmm36-jul2017.ff/merged.hdb
charmm36-jul2017.ff/ions.itp
charmm36-jul2017.ff/merged.arn
charmm36-jul2017.ff/merged.vsd
charmm36-jul2017.ff/spce.itp
charmm36-jul2017.ff/old_c36_cmap.itp
charmm36-jul2017.ff/ffnonbonded.itp
charmm36-jul2017.ff/atomtypes.atp
charmm36-jul2017.ff/merged.c.tdb
charmm36-jul2017.ff/forcefield.itp
charmm36-jul2017.ff/watermodels.dat
charmm36-jul2017.ff/tip4p.itp
charmm36-jul2017.ff/cmap.itp
charmm36-jul2017.ff/merged.n.tdb
charmm36-jul2017.ff/tip3p.itp
charmm36-jul2017.ff/merged.r2b


In [26]:
# Create system topology
# Import module
from biobb_gromacs.gromacs.pdb2gmx import pdb2gmx

# Create inputs/outputs
output_pdb2gmx_gro     = str(MD_dir / f'{ab}_pdb2gmx.gro')
output_pdb2gmx_top_zip = str(MD_dir / f'{ab}_pdb2gmx_top.zip')

prop = {
    'force_field' : 'charmm36-jul2017',
    'gmx_lib' : MD_dir,
    'water_type' : 'tip3p',
    'ignh' : True
}

# Create and launch bb
pdb2gmx(input_pdb_path=fixed_pdb, 
        output_gro_path=output_pdb2gmx_gro, 
        output_top_zip_path=output_pdb2gmx_top_zip,
        properties=prop
)

2026-05-14 12:22:47,067 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.pdb2gmx Version: 5.2.1
2026-05-14 12:22:47,068 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d
2026-05-14 12:22:47,068 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_clean.pdb --> sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d
2026-05-14 12:22:47,069 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d/4G6K_HL_clean.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d/4G6K_HL_pdb2gmx.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp -ignh
2026-05-14 12:22:47,291 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright pdb2gmx -f

2026-05-14 12:22:47,291 [MainThread  ] [INFO ]                 :-) GROMACS - gmx pdb2gmx, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d/4G6K_HL_clean.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ea7e5e0e-1927-44c3-bdf5-953c6f5d5d4d/4G6K_HL_pdb2gmx.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp -ignh

Opening force field file /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/merged.r2b
there were 70 atoms with zero occupancy and 0 atoms with          occupancy unequal to one (out of 3291 atoms). Check your pdb

2026-05-14 12:22:47,292 [MainThread  ] [INFO ]  Compressing topology to: data/bio2/3_MD/4G6K_HL_pdb2gmx_top.zip
2026-05-14 12:22:47,317 [MainThread  ] [INFO ]  Adding:
2026-05-14 12:22:47,318 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/cmap.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffnonbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/forcefield.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/gb.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ions.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibod

0

<a id="box"></a>
***
### Create solvent box
Define the unit cell for the **protein structure MD system** to fill it with water molecules.<br>
A **cubic box** is used to define the unit cell, with a **distance from the protein to the box edge of 1.0 nm**. The protein is **centered in the box**.  

***
**Building Blocks** used:
 - [Editconf](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.editconf) from **biobb_gromacs.gromacs.editconf** 
***

In [27]:
# Editconf: Create solvent box
# Import module
from biobb_gromacs.gromacs.editconf import editconf

# Create prop dict and inputs/outputs
output_editconf_gro = str(MD_dir / f'{ab}_editconf.gro')

prop = {
    'box_type': 'dodecahedron',
    'distance_to_molecule': 0.7,
    'center_molecule': False
}

#Create and launch bb
editconf(input_gro_path=output_pdb2gmx_gro, 
         output_gro_path=output_editconf_gro,
         properties=prop
)

2026-05-14 12:22:49,398 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.editconf Version: 5.2.1
2026-05-14 12:22:49,399 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056
2026-05-14 12:22:49,399 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_pdb2gmx.gro --> sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056
2026-05-14 12:22:49,400 [MainThread  ] [INFO ]  Distance of the box to molecule:   0.70
2026-05-14 12:22:49,400 [MainThread  ] [INFO ]  Box type: dodecahedron
2026-05-14 12:22:49,400 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright editconf -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056/4G6K_HL_pdb2gmx.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056/4G6K_HL_editc

2026-05-14 12:22:49,430 [MainThread  ] [INFO ]                 :-) GROMACS - gmx editconf, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright editconf -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056/4G6K_HL_pdb2gmx.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056/4G6K_HL_editconf.gro -bt dodecahedron -d 0.7


GROMACS reminds you: "What If None Of Your Dreams Come True ?" (E. Costello)




2026-05-14 12:22:49,430 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_673ee17c-afe4-4dee-8149-cb8bdb24f056']


0

<a id="water"></a>
***
### Fill the box with water molecules
Fill the unit cell for the **protein structure system** with water molecules.<br>
The solvent type used is the default **Simple Point Charge water (SPC)**, a generic equilibrated 3-point solvent model. 

***
**Building Blocks** used:
 - [Solvate](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.solvate) from **biobb_gromacs.gromacs.solvate** 
***

In [28]:
# Solvate: Fill the box with water molecules
from biobb_gromacs.gromacs.solvate import solvate

# Create prop dict and inputs/outputs
output_solvate_gro     = str(MD_dir / f'{ab}_solvate.gro')
output_solvate_top_zip = str(MD_dir / f'{ab}_solvate_top.zip')

# Create and launch bb
solvate(input_solute_gro_path=output_editconf_gro, 
        output_gro_path=output_solvate_gro, 
        input_top_zip_path=output_pdb2gmx_top_zip, 
        output_top_zip_path=output_solvate_top_zip,
)

2026-05-14 12:22:51,618 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.solvate Version: 5.2.1
2026-05-14 12:22:51,619 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e
2026-05-14 12:22:51,619 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_editconf.gro --> sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e
2026-05-14 12:22:51,625 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_pdb2gmx_top.zip
2026-05-14 12:22:51,626 [MainThread  ] [INFO ]  to:
2026-05-14 12:22:51,626 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/cmap.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/ffbonded.itp', '/home/rchaves/repo/biob

2026-05-14 12:22:51,770 [MainThread  ] [INFO ]                 :-) GROMACS - gmx solvate, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright solvate -cp /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/4G6K_HL_editconf.gro -cs spc216.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/4G6K_HL_solvate.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/p2g.top

Reading solute configuration
Reading solvent configuration

Initialising inter-atomic distances...
Generating solvent configuration
Will generate new solvent configuration of 6x6x4 boxes

2026-05-14 12:22:51,773 [MainThread  ] [INFO ]  Compressing topology to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_ca7d3d74-ce22-4c58-9139-cc27b612d40e/4G6K_HL_solvate_top.zip
2026-05-14 12:22:51,805 [MainThread  ] [INFO ]  Adding:
2026-05-14 12:22:51,809 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/cmap.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffnonbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/forcefield.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/gb.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/

0

#### Visualizing 3D structure
Visualizing the **protein system** with the newly added **solvent box** using **NGL**.<br> Note the **cubic box** filled with **water molecules** surrounding the **protein structure**, which is **centered** right in the middle of the cube.

In [5]:
# Show protein
view = nv.show_structure_file(output_solvate_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='green')
view.add_representation(repr_type='ball+stick', selection='SOL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

<a id="ions"></a>
***
### Adding ions
Add ions to neutralize the **protein structure** charge
- [Step 1](#ionsStep1): Creating portable binary run file for ion generation
- [Step 2](#ionsStep2): Adding ions to **neutralize** the system
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Genion](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.genion) from **biobb_gromacs.gromacs.genion** 
***

<a id="ionsStep1"></a>
#### Step 1: Creating portable binary run file for ion generation
A simple **energy minimization** molecular dynamics parameters (mdp) properties will be used to generate the portable binary run file for **ion generation**, although **any legitimate combination of parameters** could be used in this step.

In [30]:
# Grompp: Creating portable binary run file for ion generation
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppion_tpr = str(MD_dir / f'{ab}_gppion.tpr')
prop = {
    'simulation_type': 'ions',
    'gmx_lib' : MD_dir,
    'maxwarn': 1
}

# Create and launch bb
grompp(input_gro_path=output_solvate_gro, 
       input_top_zip_path=output_solvate_top_zip, 
       output_tpr_path=output_gppion_tpr,
       properties=prop
)

2026-05-14 12:22:57,115 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-14 12:22:57,116 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad
2026-05-14 12:22:57,116 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_solvate.gro --> sandbox_8b593d26-43be-4463-85ea-fd59a5570fad
2026-05-14 12:22:57,123 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_solvate_top.zip
2026-05-14 12:22:57,123 [MainThread  ] [INFO ]  to:
2026-05-14 12:22:57,123 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/cmap.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/ffbonded.itp', '/home/rchaves/repo/biobb/

2026-05-14 12:23:01,662 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/4G6K_HL_solvate.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/4G6K_HL_solvate.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/s

2026-05-14 12:23:01,676 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8b593d26-43be-4463-85ea-fd59a5570fad']
2026-05-14 12:23:01,676 [MainThread  ] [INFO ]  


0

<a id="ionsStep2"></a>
#### Step 2: Adding ions to neutralize the system
Replace **solvent molecules** with **ions** to **neutralize** the system.

In [6]:
# Genion: Adding ions to neutralize the system
from biobb_gromacs.gromacs.genion import genion

# Create prop dict and inputs/outputs
output_genion_gro     = str(MD_dir / f'{ab}_genion.gro')
output_genion_top_zip = str(MD_dir / f'{ab}_genion_top.zip')
prop={
    'neutral':True,
    'ionic_concentration' : 0.15
}

# Create and launch bb
genion(input_tpr_path=output_gppion_tpr, 
       output_gro_path=output_genion_gro, 
       input_top_zip_path=output_solvate_top_zip, 
       output_top_zip_path=output_genion_top_zip, 
       properties=prop
)

2026-05-14 11:35:44,118 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.genion Version: 5.2.1
2026-05-14 11:35:44,119 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb
2026-05-14 11:35:44,119 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_gppion.tpr --> sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb
2026-05-14 11:35:44,121 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/67985022-5bb1-43d2-b148-e0003cfde2a7.stdin --> sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb
2026-05-14 11:35:44,126 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_solvate_top.zip
2026-05-14 11:35:44,126 [MainThread  ] [INFO ]  to:
2026-05-14 11:35:44,126 [MainThread  ] [INFO ]  ['1abc8d5e-1185-4089-89b7-ba7ff907daba/cmap.itp', '1abc8d5e-11

2026-05-14 11:35:44,184 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright genion -s /home/rchaves/repo/biobb/biobb_wf_antibody/...' finalized with exit code 0
2026-05-14 11:35:44,184 [MainThread  ] [INFO ]  Will try to add 0 NA ions and 7 CL ions.
Select a continuous group of solvent molecules
Selected 13: 'SOL'

Processing topology
Replacing 7 solute molecules in topology file (1abc8d5e-1185-4089-89b7-ba7ff907daba/p2g.top)  by 0 NA and 7 CL ions.



/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/biobb_common/generic/biobb_object.py:187: UserWarning: Warning: ionic_concentration is not a recognized property. The most similar property is: concentration
  warnings.warn(
2026-05-14 11:35:44,184 [MainThread  ] [INFO ]                  :-) GROMACS - gmx genion, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright genion -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb/4G6K_HL_gppion.tpr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb/4G6K_HL_genion.gro -p 1abc8d5e-1185-4089-89b7-ba7ff907daba/p2g.top -neutral -seed 1993

Reading file 

2026-05-14 11:35:44,186 [MainThread  ] [INFO ]  Compressing topology to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c4689e1e-345c-4de0-9367-f0b796becbcb/4G6K_HL_genion_top.zip
2026-05-14 11:35:44,213 [MainThread  ] [INFO ]  Adding:
2026-05-14 11:35:44,213 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/cmap.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/ffnonbonded.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/forcefield.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/charmm36-jul2017.ff/gb.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/b

0

#### Visualizing 3D structure
Visualizing the **neutralized protein system** with the newly added **ions** using **NGL**

In [40]:
# Show protein
view = nv.show_structure_file(output_genion_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='sstruc')
view.add_representation(repr_type='ball+stick', selection='NA')
view.add_representation(repr_type='ball+stick', selection='CL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NameError: name 'output_genion_gro' is not defined

<a id="min"></a>
***
### Energetically minimize the system
Energetically minimize the **protein system** till reaching a desired potential energy.
- [Step 1](#emStep1): Creating portable binary run file for energy minimization
- [Step 2](#emStep2): Energetically minimize the **system** till reaching a force of 500 kJ mol-1 nm-1.
- [Step 3](#emStep3): Checking **energy minimization** results. Plotting energy by time during the **minimization** process.
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXEnergy](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_energy) from **biobb_analysis.gromacs.gmx_energy** 
***

<a id="emStep1"></a>
#### Step 1: Creating portable binary run file for energy minimization
The **minimization** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **energy minimization**:

-  integrator  = steep ; Algorithm (steep = steepest descent minimization)
-  emtol       = 1000.0 ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
-  emstep      = 0.01 ; Minimization step size (nm)
-  nsteps      = 50000 ; Maximum number of (minimization) steps to perform

In this particular example, the method used to run the **energy minimization** is the default **steepest descent**, but the **maximum force** is placed at **500 KJ/mol\*nm^2**, and the **maximum number of steps** to perform (if the maximum force is not reached) to **5,000 steps**. 

In [ ]:
# https://github.com/alevil-gmx/workflow_template/blob/main/workflow_template/data/input/emin-charmm.mdp
input_mdp_min = MD_dir / "emin-charmm.mdp"
fl = """title       = CHARMM steepest descent enrgy minimisation

; Parameters describing what to do, when to stop and what to save
integrator  = steep  ; Algorithm (steep = steepest descent minimization)
emtol       = 1000.0 ; Stop minimization when the maximum force < 1000.0 kJ/mol/nm
emstep      = 0.01   ; Minimization step size
nstenergy   = 500    ; save energies every 1.0 ps, so we can observe if we are successful
nsteps      = -1     ; run as long as we need
; Settings that make sure we run with parameters in harmony with the selected force-field
constraints             = h-bonds   ; bonds involving H are constrained
rcoulomb                = 1.2       ; short-range electrostatic cutoff (in nm)
rvdw                    = 1.2       ; short-range van der Waals cutoff (in nm)
vdw-modifier            = Force-switch ;  specific CHARMM
rvdw_switch             = 1.0       ;
DispCorr                = EnerPres  ; account for cut-off vdW scheme
coulombtype             = PME       ; Particle Mesh Ewald for long-range electrostatics
fourierspacing          = 0.15     ; grid spacing for FFT
"""

with open(input_mdp_min, 'w') as f:
    f.write(fl)

In [ ]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppmin_tpr = MD_dir / f'{ab}_gppmin.tpr'

prop = {
    'gmx_lib' : MD_dir,
}

# Create and launch bb
grompp(input_gro_path=output_genion_gro, 
       input_top_zip_path=output_genion_top_zip, 
       input_mdp_path=input_mdp_min,
       output_tpr_path=output_gppmin_tpr,  
       properties=prop
)





 0
2026-05-12 16:15:38,740 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-12 16:15:38,741 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a
2026-05-12 16:15:38,741 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_genion.gro --> sandbox_9d332695-5628-42c8-9eae-8286064b894a
2026-05-12 16:15:38,742 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/emin-charmm.mdp --> sandbox_9d332695-5628-42c8-9eae-8286064b894a
2026-05-12 16:15:38,745 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_genion_top.zip
2026-05-12 16:15:38,745 [MainThread  ] [INFO ]  to:
2026-05-12 16:15:38,746 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/p2g.top', '/home/rcha

2026-05-12 16:15:38,747 [MainThread  ] [INFO ]  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/posre_Protein_chain_L.itp
2026-05-12 16:15:38,748 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/4G6K_HL_genion.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/4G6K_HL_genion.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/4G6K_HL_gppmin.tpr -po /home/rchave

2026-05-12 16:15:39,070 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/4G6K_HL_genion.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/4G6K_HL_genion.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/san

2026-05-12 16:15:39,072 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9d332695-5628-42c8-9eae-8286064b894a']
2026-05-12 16:15:39,072 [MainThread  ] [INFO ]  


0

<a id="emStep2"></a>
#### Step 2: Running Energy Minimization
Running **energy minimization** using the **tpr file** generated in the previous step. 

In [ ]:
# Mdrun: Running minimization
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_min_trr = MD_dir / f'{ab}_min.trr'
output_min_gro = MD_dir / f'{ab}_min.gro'
output_min_edr = MD_dir / f'{ab}_min.edr'
output_min_log = MD_dir / f'{ab}_min.log'

# Create and launch bb
mdrun(input_tpr_path=output_gppmin_tpr, 
      output_trr_path=output_min_trr, 
      output_gro_path=output_min_gro, 
      output_edr_path=output_min_edr, 
      output_log_path=output_min_log,
)

2026-05-12 16:15:53,041 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-12 16:15:53,042 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63
2026-05-12 16:15:53,042 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_gppmin.tpr --> sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63
2026-05-12 16:15:53,044 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_min.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_gppmin.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_min.gro -e /home/rchaves/repo/biobb/b

2026-05-12 16:16:35,728 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_min.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_gppmin.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_min.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63/4G6K_HL_min.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/noteb

2026-05-12 16:16:35,734 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_958e4232-0e6d-490f-8ccb-e93fe5ffab63']
2026-05-12 16:16:35,734 [MainThread  ] [INFO ]  


0

<a id="emStep3"></a>
#### Step 3: Checking Energy Minimization results
Checking **energy minimization** results. Plotting **potential energy** by time during the minimization process. 

In [ ]:
# GMXEnergy: Getting system energy by time  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
output_min_ene_xvg = MD_dir / f'{ab}_min_ene.xvg'
prop = {
    'terms':  ["Potential"]
}

# Create and launch bb
gmx_energy(input_energy_path=output_min_edr, 
          output_xvg_path=output_min_ene_xvg, 
          properties=prop
)

2026-05-12 16:18:16,959 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-05-12 16:18:16,960 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c
2026-05-12 16:18:16,960 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_min.edr --> sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c
2026-05-12 16:18:16,961 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c/4G6K_HL_min.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c/4G6K_HL_min_ene.xvg -xvg none < /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a

2026-05-12 16:18:16,973 [MainThread  ] [INFO ]  Command 'gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/noteb...' finalized with exit code 0
2026-05-12 16:18:16,974 [MainThread  ] [INFO ]  
Statistics over 1705 steps [ 0.0000 through 1704.0000 ps ], 1 data sets
All statistics are over 1350 points (frames)

Energy                      Average   Err.Est.       RMSD  Tot-Drift
-------------------------------------------------------------------------------
Potential                -1.65949e+06      25000      66142    -164964  (kJ/mol)



2026-05-12 16:18:16,974 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c/4G6K_HL_min.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c/4G6K_HL_min_ene.xvg -xvg none

Opened /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c/4G6K_HL_min.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
------

2026-05-12 16:18:16,975 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76cd01d9-166f-4595-a1e2-db5d5ae99c3c']
2026-05-12 16:18:16,975 [MainThread  ] [INFO ]  


0

In [ ]:
import plotly
import plotly.graph_objs as go

#Read data from file and filter energy values higher than 1000 Kj/mol^-1
with open(output_min_ene_xvg,'r') as energy_file:
    x,y = map(
        list,
        zip(*[
            (float(line.split()[0]),float(line.split()[1]))
            for line in energy_file 
            if not line.startswith(("#","@")) 
            if float(line.split()[1]) < 1000 
        ])
    )

plotly.offline.init_notebook_mode(connected=True)

fig = {
    "data": [go.Scatter(x=x, y=y)],
    "layout": go.Layout(title="Energy Minimization",
                        xaxis=dict(title = "Energy Minimization Step"),
                        yaxis=dict(title = "Potential Energy KJ/mol-1")
                       )
}

plotly.offline.iplot(fig)

<a id="npt"></a>
***
### Equilibrate the system (NPT)
Equilibrate the **protein system** in **NPT** ensemble (constant Number of particles, Pressure and Temperature).
- [Step 1](#eqNPTStep1): Creating portable binary run file for system equilibration
- [Step 2](#eqNPTStep2): Equilibrate the **protein system** with **NPT** ensemble.
- [Step 3](#eqNPTStep3): Checking **NPT Equilibration** results. Plotting **system pressure and density** by time during the **NPT equilibration** process.
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXEnergy](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_energy) from **biobb_analysis.gromacs.gmx_energy** 
***

<a id="eqNPTStep1"></a>
#### Step 1: Creating portable binary run file for system equilibration (NPT)

The **npt** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **NPT equilibration** with **protein restraints** (see [GROMACS mdp options](http://manual.gromacs.org/documentation/2018/user-guide/mdp-options.html)):

-  Define                   = -DPOSRES
-  integrator               = md
-  dt                       = 0.002
-  nsteps                   = 5000
-  pcoupl = Parrinello-Rahman
-  pcoupltype = isotropic
-  tau_p = 1.0
-  ref_p = 1.0
-  compressibility = 4.5e-5
-  refcoord_scaling = com
-  gen_vel = no

In this particular example, the default parameters will be used: **md** integrator algorithm, a **time step** of **2fs**, **5,000 equilibration steps** with the protein **heavy atoms restrained**, and a Parrinello-Rahman **pressure coupling** algorithm.

*Please note that for the sake of time this tutorial is only running 10ps of NPT equilibration, whereas in the [original example](http://www.mdtutorials.com/gmx/lysozyme/07_equil2.html) the simulated time was 100ps.*

In [ ]:
# From https://github.com/alevil-gmx/workflow_template/blob/main/workflow_template/data/input/md_eq_posre_charmm36m.mdp
# !! se cambia define -> Define y ns_type -> ns-type o el biobb da error (TO-DO arreglar)

input_mdp_eq = MD_dir / "md_eq_posre_charmm36m.mdp"
fl="""Define                   = -DPOSRES
integrator               = md
tinit                    = 0
dt                       = 0.002
nsteps                   = 500000   ; 1  ns
comm_grps                = system
nstxout                  = 0
nstvout                  = 0
nstfout                  = 0
nstlog                   = 10000
nstenergy                = 10000
nstxout-compressed       = 10000
compressed-x-grps        = system 
nstlist                  = 10
ns-type                  = grid
pbc                      = xyz
; 
coulombtype              = PME
rcoulomb                 = 1.2
fourierspacing           = 0.15 
;
vdwtype                  = Cut-off
vdw_modifier=Force-switch
rvdw_switch              = 1.0
rvdw                     = 1.2
DispCorr                 = EnerPres
;
constraints              = h-bonds
constraint_algorithm     = LINCS
; new
Pcoupl                   = Berendsen
tau_p                    = 5.0  
ref_p                    = 1.0 
compressibility          = 4.5e-5 
refcoord-scaling         = all     ; to use with pos restrain
Tcoupl                   = v-rescale 
tc-grps                  = system
tau_t                    = 0.5 
ref_t                    = 298 """
with open(input_mdp_eq, 'w') as f:
    f.write(fl)

In [ ]:
# Grompp: Creating portable binary run file for NPT Equilibration
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
output_gppnpt_tpr = MD_dir / f'{ab}_gppnpt.tpr'

prop = {
    'mdp': {
        'nsteps' : 5000, # subir
    },
    'gmx_lib' : MD_dir,
    'maxwarn' : 1,
}

# Create and launch bb
grompp(input_gro_path=output_min_gro, 
       input_top_zip_path=output_genion_top_zip, 
       input_mdp_path=input_mdp_eq,
       output_tpr_path=output_gppnpt_tpr,  
       properties=prop
)





 1
2026-05-12 16:20:02,311 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-12 16:20:02,312 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1
2026-05-12 16:20:02,312 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_min.gro --> sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1


2026-05-12 16:20:02,314 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/md_eq_posre_charmm36m.mdp --> sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1
2026-05-12 16:20:02,317 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_genion_top.zip
2026-05-12 16:20:02,317 [MainThread  ] [INFO ]  to:
2026-05-12 16:20:02,317 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/p2g.top', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/p2g_Protein_chain_H.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/p2g_Protein_chain_L.itp', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/posre_Protein_chain_H.itp', '/home/rchaves/repo

2026-05-12 16:20:02,711 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/4G6K_HL_min.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/4G6K_HL_min.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_8

2026-05-12 16:20:02,714 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_895ab682-ee5a-49f9-b937-a2ba4662cff1']
2026-05-12 16:20:02,714 [MainThread  ] [INFO ]  


0

<a id="eqNPTStep2"></a>
#### Step 2: Running NPT equilibration

In [ ]:
# Mdrun: Running NPT System Equilibration
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_npt_trr = MD_dir / f'{ab}_npt.trr'
output_npt_gro = MD_dir / f'{ab}_npt.gro'
output_npt_edr = MD_dir / f'{ab}_npt.edr'
output_npt_log = MD_dir / f'{ab}_npt.log'
output_npt_cpt = MD_dir / f'{ab}_npt.cpt'

# Create and launch bb
mdrun(input_tpr_path=output_gppnpt_tpr, 
      output_trr_path=output_npt_trr, 
      output_gro_path=output_npt_gro, 
      output_edr_path=output_npt_edr, 
      output_log_path=output_npt_log, 
      output_cpt_path=output_npt_cpt,
)

2026-05-12 16:20:25,293 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-12 16:20:25,294 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae
2026-05-12 16:20:25,294 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_gppnpt.tpr --> sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae
2026-05-12 16:20:25,296 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_npt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_gppnpt.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_npt.gro -e /home/rchaves/repo/biobb/b

2026-05-12 16:21:59,388 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/b...' finalized with exit code 0


2026-05-12 16:21:59,389 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_npt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_gppnpt.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_npt.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae/4G6K_HL_npt.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/noteb

2026-05-12 16:21:59,395 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9ada218d-0590-425d-b9df-9e1550ae5fae', 'traj_comp.xtc']
2026-05-12 16:21:59,396 [MainThread  ] [INFO ]  Path data/bio2/3_MD/4G6K_HL_npt.trr --- biobb_gromacs.gromacs.mdrun: Unexisting output_trr_path file.
2026-05-12 16:21:59,396 [MainThread  ] [INFO ]  


/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/biobb_common/tools/file_utils.py:801: UserWarning: Path data/bio2/3_MD/4G6K_HL_npt.trr --- biobb_gromacs.gromacs.mdrun: Unexisting output_trr_path file.
  warnings.warn(not_found_error_string)


0

<a id="eqNPTStep3"></a>
#### Step 3: Checking NPT Equilibration results
Checking **NPT Equilibration** results. Plotting **system pressure and density** by time during the **NPT equilibration** process. 

In [ ]:
# GMXEnergy: Getting system pressure and density by time during NPT Equilibration  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
output_npt_pd_xvg = MD_dir / f'{ab}_npt_PD.xvg'
prop = {
    'terms':  ["Pressure","Density"]
}

# Create and launch bb
gmx_energy(input_energy_path=output_npt_edr, 
          output_xvg_path=output_npt_pd_xvg, 
          properties=prop
)

2026-05-12 16:44:04,010 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-05-12 16:44:04,011 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c
2026-05-12 16:44:04,011 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_npt.edr --> sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c
2026-05-12 16:44:04,012 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c/4G6K_HL_npt.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c/4G6K_HL_npt_PD.xvg -xvg none < /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bd

2026-05-12 16:44:04,024 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c/4G6K_HL_npt.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c/4G6K_HL_npt_PD.xvg -xvg none

Opened /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c/4G6K_HL_npt.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.
-------

2026-05-12 16:44:04,025 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3bcefc84-469c-44fe-bdcc-e5b93377895c']
2026-05-12 16:44:04,026 [MainThread  ] [INFO ]  


0

In [ ]:
import plotly
from plotly import subplots
import plotly.graph_objs as go

# Read pressure and density data from file 
with open(output_npt_pd_xvg,'r') as pd_file:
    x,y,z = map(
        list,
        zip(*[
            (float(line.split()[0]),float(line.split()[1]),float(line.split()[2]))
            for line in pd_file 
            if not line.startswith(("#","@")) 
        ])
    )

plotly.offline.init_notebook_mode(connected=True)

trace1 = go.Scatter(
    x=x,y=y
)
trace2 = go.Scatter(
    x=x,y=z
)

fig = subplots.make_subplots(rows=1, cols=2, print_grid=False)

fig.append_trace(trace1, 1, 1)
fig.append_trace(trace2, 1, 2)

fig['layout']['xaxis1'].update(title='Time (ps)')
fig['layout']['xaxis2'].update(title='Time (ps)')
fig['layout']['yaxis1'].update(title='Pressure (bar)')
fig['layout']['yaxis2'].update(title='Density (Kg*m^-3)')

fig['layout'].update(title='Pressure and Density during NPT Equilibration')
fig['layout'].update(showlegend=False)

plotly.offline.iplot(fig)

<a id="free"></a>
***
### Free Molecular Dynamics Simulation
Upon completion of the **two equilibration phases (NVT and NPT)**, the system is now well-equilibrated at the desired temperature and pressure. The **position restraints** can now be released. The last step of the **protein** MD setup is a short, **free MD simulation**, to ensure the robustness of the system. 
- [Step 1](#mdStep1): Creating portable binary run file to run a **free MD simulation**.
- [Step 2](#mdStep2): Run short MD simulation of the **protein system**.
- [Step 3](#mdStep3): Checking results for the final step of the setup process, the **free MD run**. Plotting **Root Mean Square deviation (RMSd)** and **Radius of Gyration (Rgyr)** by time during the **free MD run** step. 
***
**Building Blocks** used:
 - [Grompp](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.grompp) from **biobb_gromacs.gromacs.grompp** 
 - [Mdrun](https://biobb-md.readthedocs.io/en/latest/gromacs.html#module-gromacs.mdrun) from **biobb_gromacs.gromacs.mdrun** 
 - [GMXRms](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_rms) from **biobb_analysis.gromacs.gmx_rms** 
 - [GMXRgyr](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_rgyr) from **biobb_analysis.gromacs.gmx_rgyr** 
***

<a id="mdStep1"></a>
#### Step 1: Creating portable binary run file to run a free MD simulation

The **free** type of the **molecular dynamics parameters (mdp) property** contains the main default parameters to run an **free MD simulation** (see [GROMACS mdp options](http://manual.gromacs.org/documentation/2018/user-guide/mdp-options.html)):

-  integrator               = md
-  dt                       = 0.002 (ps)
-  nsteps                   = 50000

In this particular example, the default parameters will be used: **md** integrator algorithm, a **time step** of **2fs**, and a total of **50,000 md steps** (100ps).

*Please note that for the sake of time this tutorial is only running 100ps of free MD, whereas in the [original example](http://www.mdtutorials.com/gmx/lysozyme/08_MD.html) the simulated time was 1ns (1000ps).*

In [ ]:
# https://github.com/alevil-gmx/workflow_template/blob/main/workflow_template/data/input/md_charmm36m.mdp
# ns_type -> ns-type
input_mdp_md = MD_dir / "md_charmm36m.mdp"

file=""";define                   =  -DPOSRES
;include                  = -I/home/villa/work/UseCaseI/forcefield
integrator               = md
tinit                    = 0
dt                       = 0.002
nsteps                   = 50000000   ; 100  ns
;nstcomm                  = 1
comm_grps                = system
nstxout                  = 0
nstvout                  = 0
nstfout                  = 0
nstlog                   = 100000
nstenergy                = 100000
nstxout-compressed       = 100000
compressed-x-grps        = system 
nstlist                  = 10
ns-type                  = grid
pbc                      = xyz
; neighbour
; rlist                   = 1.2 ; not used when cutoff-scheme = verlet
cutoff-scheme           = Verlet
; coulomb
coulombtype              = PME
rcoulomb                 = 1.2
fourierspacing           = 0.15 ;  For constant accuracy one should keep fourier-spacing/rcoulomb constant = 0.125
; vdw
vdwtype                  = Cut-off
vdw_modifier             = Force-switch
rvdw_switch              = 1.0  ; 
rvdw                     = 1.2
DispCorr                 = EnerPres ; while for lipid bilayer,  it's a difficult topic in the CHARMM force field. If one don't have lipids bi- or monolayers one should use it. 
;
constraints              = h-bonds
constraint_algorithm     = LINCS
; barostat
Pcoupl                   = Parrinello-Rahman   
tau_p                    = 5.0  
ref_p                    = 1.0 
compressibility          = 4.5e-5 
; thermostat
Tcoupl                   = v-rescale 
tc-grps                  = system
tau_t                    = 0.5 ; or 0.1 
ref_t                    = 298 """
with open(input_mdp_md, 'w') as f:
    f.write(file)

In [ ]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs

output_gppmd_tpr = MD_dir / f"{ab}_gppmd.tpr"

prop = {
    'mdp': {
        'nsteps' :  500_000, # 1 ns
        'nstxout' :  10_000
    },
    'gmx_lib' : MD_dir,
    'maxwarn' : 1
}

# Create and launch bb
grompp(input_gro_path=output_npt_gro, 
       input_top_zip_path=output_genion_top_zip, 
       input_mdp_path=input_mdp_md,
       output_tpr_path=output_gppmd_tpr, 
       input_cpt_path=output_npt_cpt, 
       properties=prop
)





 1
2026-05-12 16:55:18,284 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1


2026-05-12 16:55:18,285 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4
2026-05-12 16:55:18,285 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_npt.gro --> sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4
2026-05-12 16:55:18,287 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_npt.cpt --> sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4
2026-05-12 16:55:18,288 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/md_charmm36m.mdp --> sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4
2026-05-12 16:55:18,290 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_genion_top.zip
2026-05-12 16:55:18,291 [MainThread  ] [INFO ]  to:
2026-05-12 16:55:18,291 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-

2026-05-12 16:55:18,725 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4/4G6K_HL_npt.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4/4G6K_HL_npt.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3

2026-05-12 16:55:18,732 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_3a1600de-c6a1-4e4a-89bf-8d00958733f4']
2026-05-12 16:55:18,733 [MainThread  ] [INFO ]  


0

<a id="mdStep2"></a>
#### Step 2: Running short free MD simulation

In [ ]:
# Mdrun: Running free dynamics
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
output_md_trr = MD_dir / f'{ab}_md.trr'
output_md_gro = MD_dir / f'{ab}_md.gro'
output_md_edr = MD_dir / f'{ab}_md.edr'
output_md_log = MD_dir / f'{ab}_md.log'
output_md_cpt = MD_dir / f'{ab}_md.cpt'

# Create and launch bb
mdrun(input_tpr_path=output_gppmd_tpr, 
      output_trr_path=output_md_trr, 
      output_gro_path=output_md_gro, 
      output_edr_path=output_md_edr, 
      output_log_path=output_md_log, 
      output_cpt_path=output_md_cpt,
)

2026-05-12 16:55:31,903 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-12 16:55:31,904 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8
2026-05-12 16:55:31,905 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_gppmd.tpr --> sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8
2026-05-12 16:55:31,906 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_md.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_gppmd.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_md.gro -e /home/rchaves/repo/biobb/biobb

2026-05-12 18:46:13,027 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/b...' finalized with exit code 0


2026-05-12 18:46:13,028 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_md.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_gppmd.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_md.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8/4G6K_HL_md.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks

2026-05-12 18:46:13,037 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0d286201-e0e8-4bdd-9d4a-1b598b6aa2b8', 'traj_comp.xtc']
2026-05-12 18:46:13,038 [MainThread  ] [INFO ]  


0

<a id="post"></a>
***
### Post-processing and Visualizing resulting 3D trajectory
Post-processing and Visualizing the **protein system** MD setup **resulting trajectory** using **NGL**
- [Step 1](#ppStep1): *Imaging* the resulting trajectory, **stripping out water molecules and ions** and **correcting periodicity issues**.
- [Step 2](#ppStep2): Generating a *dry* structure, **removing water molecules and ions** from the final snapshot of the MD setup pipeline.
- [Step 3](#ppStep3): Visualizing the *imaged* trajectory using the *dry* structure as a **topology**. 
***
**Building Blocks** used:
 - [GMXImage](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_image) from **biobb_analysis.gromacs.gmx_image** 
 - [GMXTrjConvStr](https://biobb-analysis.readthedocs.io/en/latest/gromacs.html#module-gromacs.gmx_trjconv_str) from **biobb_analysis.gromacs.gmx_trjconv_str** 
***

<a id="ppStep1"></a>
#### Step 1: *Imaging* the resulting trajectory.
Stripping out **water molecules and ions** and **correcting periodicity issues**  

In [ ]:
# GMXImage: "Imaging" the resulting trajectory
#           Removing water molecules and ions from the resulting structure
from biobb_analysis.gromacs.gmx_image import gmx_image

# Create prop dict and inputs/outputs
output_imaged_traj = MD_dir / f'{ab}_imaged_traj.trr'
prop = {
    'center_selection':  'Protein',
    'output_selection': 'Protein',
    'pbc' : 'nojump',
}

# Create and launch bb
gmx_image(input_traj_path=output_md_trr,
          input_top_path=output_gppmin_tpr,
          output_traj_path=output_imaged_traj, 
          properties=prop
)

2026-05-13 10:55:24,869 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_image Version: 5.2.1
2026-05-13 10:55:24,870 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96
2026-05-13 10:55:24,870 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_md.trr --> sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96
2026-05-13 10:55:24,872 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_gppmin.tpr --> sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96
2026-05-13 10:55:24,874 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/8d285cc2-b480-4a70-b46a-1c001ec3e1cd.stdin --> sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96
2026-05-13 10:55:24,874 [MainThread  ] [INFO

2026-05-13 10:55:24,960 [MainThread  ] [INFO ]                 :-) GROMACS - gmx trjconv, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx trjconv -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96/4G6K_HL_md.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96/4G6K_HL_gppmin.tpr -fit none -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96/4G6K_HL_imaged_traj.trr -center -pbc nojump -ur compact

         only has effect in combination with -pbc mol, res or atom.
         Ignoring unitcell representation.

Will write trr: Trajectory in portable xdr format
Reading file

2026-05-13 10:55:24,962 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_e548f4dd-1569-4fb7-9739-bc93f2668c96', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/8d285cc2-b480-4a70-b46a-1c001ec3e1cd.stdin']
2026-05-13 10:55:24,963 [MainThread  ] [INFO ]  


0

<a id="ppStep2"></a>
#### Step 2: Generating the output *dry* structure.
**Removing water molecules and ions** from the resulting structure

In [ ]:
# GMXTrjConvStr: Converting and/or manipulating a structure
#                Removing water molecules and ions from the resulting structure
#                The "dry" structure will be used as a topology to visualize 
#                the "imaged dry" trajectory generated in the previous step.
from biobb_analysis.gromacs.gmx_trjconv_str import gmx_trjconv_str

# Create prop dict and inputs/outputs
output_dry_gro = MD_dir / f'{ab}_md_dry.gro'
prop = {
    'selection':  'Protein'
}

# Create and launch bb
gmx_trjconv_str(input_structure_path=output_md_gro,
                input_top_path=output_gppmd_tpr,
                output_str_path=output_dry_gro, 
                properties=prop
)

2026-05-13 10:55:27,678 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_trjconv_str Version: 5.2.1
2026-05-13 10:55:27,679 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82
2026-05-13 10:55:27,680 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_md.gro --> sandbox_79207857-9844-4e00-a660-b7b4445d4a82
2026-05-13 10:55:27,682 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_gppmd.tpr --> sandbox_79207857-9844-4e00-a660-b7b4445d4a82
2026-05-13 10:55:27,684 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/710535b7-73b1-4b8a-9785-7156c7da31d3.stdin --> sandbox_79207857-9844-4e00-a660-b7b4445d4a82
2026-05-13 10:55:27,684 [MainThread  ] 

2026-05-13 10:55:27,799 [MainThread  ] [INFO ]                 :-) GROMACS - gmx trjconv, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx trjconv -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82/4G6K_HL_md.gro -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82/4G6K_HL_gppmd.tpr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82/4G6K_HL_md_dry.gro -nocenter

Will write gro: Coordinate file in Gromos-87 format
Reading file /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82/4G6K_HL_gppmd.tpr, VERSION 2026

2026-05-13 10:55:27,801 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_79207857-9844-4e00-a660-b7b4445d4a82', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/710535b7-73b1-4b8a-9785-7156c7da31d3.stdin']
2026-05-13 10:55:27,801 [MainThread  ] [INFO ]  


0

In [ ]:
# GMXImage: "Imaging" the resulting trajectory
#           Removing water molecules and ions from the resulting structure
from biobb_analysis.gromacs.gmx_image import gmx_image

# Create prop dict and inputs/outputs
output_imaged_traj_rot = MD_dir / f'{ab}_imaged_traj_rot.trr'
prop = {
    'fit' : 'rot+trans',
    'center': False
}

# Create and launch bb
gmx_image(input_traj_path=output_imaged_traj,
          input_top_path=output_dry_gro,
          output_traj_path=output_imaged_traj_rot, 
          properties=prop
)

2026-05-13 10:55:43,191 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_image Version: 5.2.1
2026-05-13 10:55:43,192 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a
2026-05-13 10:55:43,193 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_imaged_traj.trr --> sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a
2026-05-13 10:55:43,194 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_md_dry.gro --> sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a
2026-05-13 10:55:43,194 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/4812a6ea-1a25-4ad7-8dcc-16c3e3d2ccb1.stdin --> sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a
2026-05-13 10:55:43,195 [MainThread

2026-05-13 10:55:43,232 [MainThread  ] [INFO ]                 :-) GROMACS - gmx trjconv, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx trjconv -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a/4G6K_HL_imaged_traj.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a/4G6K_HL_md_dry.gro -fit rot+trans -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a/4G6K_HL_imaged_traj_rot.trr -nocenter

Will write trr: Trajectory in portable xdr format

         that are broken across periodic boundaries, they
         cannot be made whole (or treated as whole) without
         

2026-05-13 10:55:43,233 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_bf37e395-1cbe-49e9-9247-c57eff7cd73a', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/4812a6ea-1a25-4ad7-8dcc-16c3e3d2ccb1.stdin']
2026-05-13 10:55:43,234 [MainThread  ] [INFO ]  


0

<a id="ppStep3"></a>
#### Step 3: Visualizing the generated dehydrated trajectory.
Using the **imaged trajectory** (output of the [Post-processing step 1](#ppStep1)) with the **dry structure** (output of the [Post-processing step 2](#ppStep2)) as a topology.

In [ ]:
# Show trajectory
import MDAnalysis as mda
u = mda.Universe(output_dry_gro, output_imaged_traj_rot)
view = nv.show_mdanalysis(u)
view.add_trajectory(u)
view.add_licorice()
view

NGLWidget(max_frame=5)

<a id="mdStep3"></a>
#### Step 4: Checking free MD simulation results
Checking results for the final step of the setup process, the **free MD run**. Plotting **Root Mean Square deviation (RMSd)** and **Radius of Gyration (Rgyr)** by time during the **free MD run** step. **RMSd** against the **experimental structure** (input structure of the pipeline) and against the **minimized and equilibrated structure** (output structure of the NPT equilibration step).

In [ ]:
# GMXRms: Computing Root Mean Square deviation to analyse structural stability 
#         RMSd against experimental structure (backbone atoms)   

from biobb_analysis.gromacs.gmx_rms import gmx_rms

# Create prop dict and inputs/outputs
output_rms_exp = MD_dir / f'{ab}_rms_exp.xvg'
prop = {
    'selection':  'Backbone',
    #'selection': 'non-Water'
}

# Create and launch bb
gmx_rms(input_structure_path=output_gppmin_tpr,
         input_traj_path=output_imaged_traj_rot,
         output_xvg_path=output_rms_exp, 
          properties=prop)

2026-05-13 11:08:47,025 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_rms Version: 5.2.1
2026-05-13 11:08:47,026 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295
2026-05-13 11:08:47,026 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_gppmin.tpr --> sandbox_6876b352-d159-49c9-8993-449e1dc22295
2026-05-13 11:08:47,028 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_imaged_traj_rot.trr --> sandbox_6876b352-d159-49c9-8993-449e1dc22295
2026-05-13 11:08:47,028 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/e26650f0-c7bf-46b1-b06e-cfcc2c7af026.stdin --> sandbox_6876b352-d159-49c9-8993-449e1dc22295
2026-05-13 11:08:47,028 [MainThre

2026-05-13 11:08:47,090 [MainThread  ] [INFO ]                   :-) GROMACS - gmx rms, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx rms -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295/4G6K_HL_gppmin.tpr -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295/4G6K_HL_imaged_traj_rot.trr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295/4G6K_HL_rms_exp.xvg -xvg none

Reading file /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295/4G6K_HL_gppmin.tpr, VERSION 2026.0-conda_forge (single precision)
Reading 

2026-05-13 11:08:47,091 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_6876b352-d159-49c9-8993-449e1dc22295', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/e26650f0-c7bf-46b1-b06e-cfcc2c7af026.stdin']
2026-05-13 11:08:47,092 [MainThread  ] [INFO ]  


0

In [ ]:
import plotly
import plotly.graph_objs as go

# Read RMS vs experimental structure data from file 
with open(output_rms_exp,'r') as rms_exp_file:
    x2,y2 = map(
        list,
        zip(*[
            (float(line.split()[0]),float(line.split()[1]))
            for line in rms_exp_file
            if not line.startswith(("#","@")) 
        ])
    )

trace = go.Scatter(
    x = x2,
    y = y2,
    name = 'RMSd vs exp'
)

plotly.offline.init_notebook_mode(connected=True)

fig = {
    "data": trace,
    "layout": go.Layout(title="RMSd during free MD Simulation",
                        xaxis=dict(title = "Time (ps)"),
                        yaxis=dict(title = "RMSd (nm)")
                       )
}

plotly.offline.iplot(fig)


In [ ]:
# GMXRgyr: Computing Radius of Gyration to measure the protein compactness during the free MD simulation 

from biobb_analysis.gromacs.gmx_rgyr import gmx_rgyr

# Create prop dict and inputs/outputs
output_rgyr = MD_dir / f'{ab}_rgyr.xvg'
prop = {
    'selection':  'Backbone'
}

# Create and launch bb
gmx_rgyr(input_structure_path=output_gppmin_tpr,
         input_traj_path=output_imaged_traj_rot,
         output_xvg_path=output_rgyr, 
          properties=prop)

2026-05-13 11:09:12,180 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_rgyr Version: 5.2.1
2026-05-13 11:09:12,181 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591
2026-05-13 11:09:12,181 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_gppmin.tpr --> sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591
2026-05-13 11:09:12,183 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_imaged_traj_rot.trr --> sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591
2026-05-13 11:09:12,183 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/7d5e12c8-16d5-4652-8516-2192596eae7c.stdin --> sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591
2026-05-13 11:09:12,184 [MainThr

2026-05-13 11:09:12,226 [MainThread  ] [INFO ]                  :-) GROMACS - gmx gyrate, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx gyrate -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591/4G6K_HL_gppmin.tpr -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591/4G6K_HL_imaged_traj_rot.trr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591/4G6K_HL_rgyr.xvg -xvg none

Reading file /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591/4G6K_HL_gppmin.tpr, VERSION 2026.0-conda_forge (single precision)
Readin

2026-05-13 11:09:12,227 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_83a0d6fc-f36b-46a3-b35e-322d8a958591', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/7d5e12c8-16d5-4652-8516-2192596eae7c.stdin']
2026-05-13 11:09:12,228 [MainThread  ] [INFO ]  


0

In [ ]:
import plotly
import plotly.graph_objs as go

# Read Rgyr data from file 
with open(output_rgyr,'r') as rgyr_file:
    x,y = map(
        list,
        zip(*[
            (float(line.split()[0]),float(line.split()[1]))
            for line in rgyr_file 
            if not line.startswith(("#","@")) 
        ])
    )

plotly.offline.init_notebook_mode(connected=True)

fig = {
    "data": [go.Scatter(x=x, y=y)],
    "layout": go.Layout(title="Radius of Gyration",
                        xaxis=dict(title = "Time (ps)"),
                        yaxis=dict(title = "Rgyr (nm)")
                       )
}

plotly.offline.iplot(fig)

### Clustering CDR loops

In [15]:
from anarcii import Anarcii
import shutil 
model = Anarcii(seq_type="antibody", mode="accuracy", verbose=True)
# results renunbers the file and returns the numbering
results = model.number(fixed_pdb)
anarcii_out = Path(fixed_pdb).stem.lower() + '-anarcii-imgt.pdb'
anarcii_pdb = str(MD_dir / anarcii_out)
shutil.move(anarcii_out, anarcii_pdb)

Using device CPU with 16 CPUs
Batch size: 32
	Speed is a balance of batch size and length diversity. Adjust accordingly. For a full explanation see:
 	wiki/FAQs#recommended-batch-sizes
 	Seqs all similar length (+/-5), increase batch size. Mixed lengths (+/-30), reduce.


Consider larger batch size for optimal GPU performance.

Length of sequence list: 2
Processing sequences in 1 chunks of 102400 sequences.
Processing chunk 1 of 1.

 2 Long sequences detected - running in sliding window. This is slow.
Max probability windows selected.

Making predictions on 1 batches.


PDBx model index, chain ID: 0, H
ANARCII chain type (score): H (30.586580276489258)
 Sequence length: 128
 Sequence: [((1, ' '), 'Q'), ((2, ' '), 'V'), ((3, ' '), 'Q'), ((4, ' '), 'L'), ((5, ' '), 'Q'), ((6, ' '), 'E'), ((7, ' '), 'S'), ((8, ' '), 'G'), ((9, ' '), 'P'), ((10, ' '), '-'), ((11, ' '), 'G'), ((12, ' '), 'L'), ((13, ' '), 'V'), ((14, ' '), 'K'), ((15, ' '), 'P'), ((16, ' '), 'S'), ((17, ' '), 'Q'), ((18, ' '), 'T'), ((19, ' '), 'L'), ((20, ' '), 'S'), ((21, ' '), 'L'), ((22, ' '), 'T'), ((23, ' '), 'C'), ((24, ' '), 'S'), ((25, ' '), 'F'), ((26, ' '), 'S'), ((27, ' '), 'G'), ((28, ' '), 'F'), ((29, ' '), 'S'), ((30, ' '), 'L'), ((31, ' '), 'S'), ((32, ' '), '-'), ((33, ' '), '-'), ((34, ' '), 'T'), ((35, ' '), 'S'), ((36, ' '), 'G'), ((37, ' '), 'M'), ((38, ' '), 'G'), ((39, ' '), 'V'), ((40, ' '), 'G'), ((41, ' '), 'W'), ((42, ' '), 'I'), ((43, ' '), 'R'), ((44, ' '), 'Q'), ((45, ' '), 'P'), ((46, ' '), 'S'), ((47, ' '), 'G'), ((48, ' '), 'K'), ((49, ' '), 'G'), ((50, ' '

'data/bio2/3_MD/4g6k_hl_clean-anarcii-imgt.pdb'

In [17]:
from anarcii.output_data_processing.schemes_constants import schemes

current_i = 0
regions = {1: 'FR1', 2: 'CDR1', 3: 'FR2', 4: 'CDR2', 5: 'FR3', 6: 'CDR3', 7: 'FR4'}
regions_range = {}
def region(x):
    """Convert region number to the corresping IMGT region name.
    https://www.imgt.org/IMGTScientificChart/Nomenclature/IMGT-FRCDRdefinition.html
    """
    return f"{regions.get(int(x), 'Unknown'):<4}"

n = len(schemes['imgt']['region_string'])
start = 1           # Current region start residue number (1-indexed)
current_region = 1  # Region number (1-indexed)
for resnum, region_i in enumerate(schemes['imgt']['region_string'], start=1):
    region_i = int(region_i)
    if region_i != current_region:
        regions_range[region_i-1] = f'{start}-{resnum-1}'
        print(f"{current_region} {region(current_region)}: {regions_range[region_i-1]}")
        start = resnum
        current_region = region_i
    if resnum == n-1:
        regions_range[region_i] = f'{start}-{n}'
        print(f"{current_region} {region(current_region)}: {regions_range[region_i]}")


1 FR1 : 1-26
2 CDR1: 27-38
3 FR2 : 39-55
4 CDR2: 56-65
5 FR3 : 66-104
6 CDR3: 105-117
7 FR4 : 118-128


In [18]:
# Show protein
view = nv.show_structure_file(anarcii_pdb)
# https://www.imgt.org/IMGTScientificChart/Nomenclature/IMGT-FRCDRdefinition.html
view.add_ball_and_stick(selection=regions_range[2]) #CDR1
view.add_ball_and_stick(selection=regions_range[4]) #CDR2
view.add_ball_and_stick(selection=regions_range[6]) #CDR3
view

NGLWidget()

In [19]:
# Translating the CDR residues to the original numbering in the PDB file (not needed for now)
import MDAnalysis as mda
u1  = mda.Universe(anarcii_pdb)
u2  = mda.Universe(fixed_pdb)

residsH = u1.select_atoms('chainID H').residues.resids
maskH = ((27 <= residsH) & (residsH <= 38)) | ((56 <= residsH) & (residsH <= 65)) | ((105 <= residsH) & (residsH <= 117))
residsA = u2.select_atoms('chainID H').residues.resids

residsL = u1.select_atoms('chainID L').residues.resids
maskL = ((27 <= residsL) & (residsL <= 38)) | ((56 <= residsL) & (residsL <= 65)) | ((105 <= residsL) & (residsL <= 117))
residsB = u2.select_atoms('chainID L').residues.resids

view = nv.show_structure_file(fixed_pdb, default_representation=False)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='all')
view.center()
# view._remote_call('setSize', target='Widget', args=['','600px'])
view.add_ball_and_stick(selection=f"( {', '.join(list(map(str, residsA[maskH])))} ) and :H")
view.add_ball_and_stick(selection=f"( {', '.join(list(map(str, residsB[maskL])))} ) and :L")
view

/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:479: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn(


NGLWidget()

In [135]:
from biobb_gromacs.gromacs.pdb2gmx import pdb2gmx
anarcii_gro = MD_dir / f'{ab}_anarcii.gro'
anarcii_zip = MD_dir / f'{ab}_anarcii.zip'

prop = {
    'force_field' : 'charmm36-jul2017',
    'gmx_lib' : MD_dir,
    'water_type' : 'tip3p',
}

# Create and launch bb
pdb2gmx(input_pdb_path=anarcii_pdb, 
        output_gro_path=anarcii_gro, 
        output_top_zip_path=anarcii_zip,
        properties=prop)

2026-05-14 10:38:44,454 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.pdb2gmx Version: 5.2.1
2026-05-14 10:38:44,455 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b
2026-05-14 10:38:44,456 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4g6k_hl_clean-anarcii-imgt.pdb --> sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b
2026-05-14 10:38:44,457 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b/4g6k_hl_clean-anarcii-imgt.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b/4G6K_HL_anarcii.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp
2026-05-14 10:38:44,678 [MainThread  ] [INFO ]  Command 'gmx -nobackup -no

2026-05-14 10:38:44,679 [MainThread  ] [INFO ]                 :-) GROMACS - gmx pdb2gmx, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b/4g6k_hl_clean-anarcii-imgt.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00eccb79-df16-4ca9-b23b-e8cd5e573c7b/4G6K_HL_anarcii.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp

Opening force field file data/bio2/3_MD/charmm36-jul2017.ff/merged.r2b
there were 70 atoms with zero occupancy and 0 atoms with          occupancy unequal to one (out of 3291 atoms). Check your pdb file.
Opening force field file data/bio2/3_MD/charmm36-jul2017.

2026-05-14 10:38:44,680 [MainThread  ] [INFO ]  Compressing topology to: data/bio2/3_MD/4G6K_HL_anarcii.zip
2026-05-14 10:38:44,704 [MainThread  ] [INFO ]  Adding:
2026-05-14 10:38:44,705 [MainThread  ] [INFO ]  ['data/bio2/3_MD/charmm36-jul2017.ff/cmap.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/ffbonded.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/ffnonbonded.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/forcefield.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/gb.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/ions.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/nbfix.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/old_c36_cmap.itp', 'data/bio2/3_MD/charmm36-jul2017.ff/tip3p.itp', 'p2g.top', 'p2g_Protein_chain_H.itp', 'p2g_Protein_chain_L.itp', 'posre_Protein_chain_H.itp', 'posre_Protein_chain_L.itp']
2026-05-14 10:38:44,705 [MainThread  ] [INFO ]  to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_anarcii.zip
2026-05-14 10:38:44,706 [MainThread  ] [INFO ]  Removed: [

0

In [136]:
from biobb_gromacs.gromacs.make_ndx import make_ndx

# Create prop dict and inputs/outputs
loop_ndx = MD_dir / f'{ab}_loop.ndx'

prop = { 
    'selection': 'r 27-38 | r 56-65 | r 105-117\nname 10 Loop'
}

# Create and launch bb
make_ndx(input_structure_path=anarcii_gro,
         output_ndx_path=loop_ndx,
         properties=prop)

2026-05-14 10:38:50,788 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.make_ndx Version: 5.2.1
2026-05-14 10:38:50,789 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83
2026-05-14 10:38:50,789 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/4G6K_HL_anarcii.gro --> sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83
2026-05-14 10:38:50,790 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/5eb85d2d-4c54-42fb-99c4-19ec6da49fa6.stdin --> sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83
2026-05-14 10:38:50,791 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright make_ndx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83/4G6K_HL_anarcii.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/n

2026-05-14 10:38:50,812 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright make_ndx -f /home/rchaves/repo/biobb/biobb_wf_antibod...' finalized with exit code 0
2026-05-14 10:38:50,812 [MainThread  ] [INFO ]  Going to read 0 old index file(s)
Analysing residue names:
There are:   432    Protein residues
Analysing Protein...

  0 System              :  6525 atoms
  1 Protein             :  6525 atoms
  2 Protein-H           :  3293 atoms
  3 C-alpha             :   432 atoms
  4 Backbone            :  1296 atoms
  5 MainChain           :  1726 atoms
  6 MainChain+Cb        :  2126 atoms
  7 MainChain+H         :  2139 atoms
  8 SideChain           :  4386 atoms
  9 SideChain-H         :  1567 atoms

 nr : group      '!': not  'name' nr name   'splitch' nr    Enter: list groups
 'a': atom       '&': and  'del' nr         'splitres' nr   'l': list residues
 't': atom type  '|': or   'keep' nr        'splitat' nr    'h': help
 'r': residue              'res' nr         'chain' char

2026-05-14 10:38:50,812 [MainThread  ] [INFO ]                 :-) GROMACS - gmx make_ndx, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright make_ndx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83/4G6K_HL_anarcii.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83/4G6K_HL_loop.ndx


Reading structure file

GROMACS reminds you: "According to my computations we're overdue for a transformation." (Jackson Browne)




2026-05-14 10:38:50,813 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9adf1782-f3c6-42b0-ad3f-fcf6639eee83', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/5eb85d2d-4c54-42fb-99c4-19ec6da49fa6.stdin']
2026-05-14 10:38:50,814 [MainThread  ] [INFO ]  


0

In [22]:
from biobb_analysis.gromacs.gmx_cluster import gmx_cluster

output_pdb_cluster = MD_dir / f'{ab}_clusters.pdb'

prop = {
    'fit_selection': 'Loop',
    'output_selection': 'System',
#    'method': 'gromos',
    'method' : 'jarvis-patrick',
    'cutoff' : 0.01,
    'nofit' : True,
}
gmx_cluster(input_structure_path=output_gppmin_tpr,
            input_traj_path=output_imaged_traj_rot,
            input_index_path=loop_ndx,
            output_pdb_path=output_pdb_cluster,
            properties=prop)

2026-05-20 12:16:36,687 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_cluster Version: 5.2.1
2026-05-20 12:16:36,688 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417
2026-05-20 12:16:36,689 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_gppmin.tpr --> sandbox_a3753338-4a12-4dc9-94bc-642a83d40417
2026-05-20 12:16:36,693 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_imaged_traj_rot.trr --> sandbox_a3753338-4a12-4dc9-94bc-642a83d40417
2026-05-20 12:16:36,694 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/3_MD/4G6K_HL_loop.ndx --> sandbox_a3753338-4a12-4dc9-94bc-642a83d40417
2026-05-20 12:16:36,695 [MainThread  ] [

2026-05-20 12:16:36,752 [MainThread  ] [INFO ]  Command 'gmx cluster -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/note...' finalized with exit code 0
2026-05-20 12:16:36,753 [MainThread  ] [INFO ]  Selected 10: 'Loop'
Selected 0: 'System'



2026-05-20 12:16:36,754 [MainThread  ] [INFO ]                 :-) GROMACS - gmx cluster, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx cluster -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417/c2f84c9d-441b-43a6-9a12-6b02ddccd89ccluster.log -dist /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417/d27933f1-bbd9-43d5-b0e3-6a8247791a3crmsd-dist.xvg -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417/1c52effa-77df-437f-afc4-dda1922f74ccrmsd-clust.xpm -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417/4G6K

2026-05-20 12:16:36,758 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_a3753338-4a12-4dc9-94bc-642a83d40417', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/082f4c01-9247-4e62-81d0-35b76652ecc9.stdin']
2026-05-20 12:16:36,758 [MainThread  ] [INFO ]  


0

In [25]:
# visualizar
view = nv.show_structure_file(str(output_pdb_cluster))
view

NGLWidget()

In [138]:
# Feed the cluster to HADDOCK to see if we can get better results than with the original structure
# ...

## Enhace sampling of CDR loops in presence of antigen with AWH

Objective: sampling the CDR loops in presence of the antigen to capture possible induced fit effects and improve docking results. To capture the whole range of conformational changes of the CDR loops, we will use the **Adaptive Weighted Histogram (AWH)** enhanced sampling method implemented in GROMACS.

In [74]:
AWH_dir = out_path / '4_AWH'
AWH_dir.mkdir(parents=True, exist_ok=True)

### Prepare topology

In [27]:
haddock_best_pdb = str(out_path / '2_dock' / 'run' / '10_seletopclusts' / 'cluster_1_model_1.pdb')
ref_pdb_src = str(out_path / '0_base' / f'{ref}.pdb')
resmap = {'data/bio2/0_base/4G6K_HL.pdb': {'H': 120, 'L': 107}}

In [28]:
import MDAnalysis as mda
from MDAnalysis.analysis import align
ref_u = mda.Universe(ref_pdb_src)
ref_HV = ref_u.select_atoms('chainID H and resnum 1-120')
ref_LV = ref_u.select_atoms('chainID L and resnum 1-107')
ref_Fv = ref_HV + ref_LV
best = mda.Universe(haddock_best_pdb)
best_Fv = best.select_atoms('chainID A and not element H')
best_HV = best_Fv.select_atoms('resnum 1-120')
best_LV = best_Fv.select_atoms('resnum 121-227')

# Align heavy atoms of the Fv
align.alignto(best_Fv, ref_Fv, match_atoms=False, weights=None)

# Align J regions backbone
align.alignto(
    mobile=best_Fv,
    reference=ref_Fv,
    select={
        'mobile': '(resnum 120 or resnum 227) and backbone',
        'reference': '(chainID H and resnum 120) or (chainID L and resnum 107) and backbone'
    },
    match_atoms=False, weights=None
)

align_pdb = str(AWH_dir / 'haddock_best_aligned.pdb')
with mda.coordinates.PDB.PDBWriter(align_pdb) as W:
    W.write(best.select_atoms('chainID A'))

# Copy coordinates by atom identity so residue-local atom order differences do not matter.
for ref_residue, best_residue in zip(ref_HV.residues, best_HV.residues):
    best_atoms_by_name = {atom.name: atom for atom in best_residue.atoms}
    ref_atoms = ref_residue.atoms
    ref_atoms.positions = [best_atoms_by_name[atom.name].position for atom in ref_atoms]

for ref_residue, best_residue in zip(ref_LV.residues, best_LV.residues):
    best_atoms_by_name = {atom.name: atom for atom in best_residue.atoms}
    ref_atoms = ref_residue.atoms
    ref_atoms.positions = [best_atoms_by_name[atom.name].position for atom in ref_atoms]

merged_pdb = str(AWH_dir / 'haddock_best_merged.pdb')
ref_u.atoms.write(merged_pdb)

/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:885: UserWarning: Unit cell dimensions not found. CRYST1 record set to unitary values.
  warnings.warn(
/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/MDAnalysis/coordinates/PDB.py:1282: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn(


In [29]:
# Before the aligment
view1 = nv.show_structure_file(ab_pdb_src)
view1.add_component(haddock_best_pdb)
view1.remove_cartoon(component=1)
view1.add_representation(repr_type='cartoon', selection=':A', component=1, color='green')
view1.center()
view1.layout.margin = "auto"
view1._remote_call('setSize', target='Widget', args=['500px','300px'])

# After the aligment
view2 = nv.show_structure_file(ab_pdb_src)
view2.add_representation(repr_type='ball+stick', selection=':H and 120-121', component=0, color='red')
view2.add_representation(repr_type='ball+stick', selection=':L and 107', component=0, color='blue')
view2.add_component(align_pdb)
view2.remove_cartoon(component=1)
view2.add_representation(repr_type='cartoon', selection=':A', component=1, color='green')
view2.add_representation(repr_type='ball+stick', selection=':A and ( 120, 227 )', component=1, color='green')
view2.center()
view2._remote_call('setSize', target='Widget', args=['500px','300px'])
view2.layout.margin = "auto"

# Merged structure
view3 = nv.show_structure_file(merged_pdb)
view3.add_component(ref_pdb_src, color='grey')
view3.remove_cartoon(component=1)
view3.add_representation(component=1, repr_type='cartoon', color='grey')
view3.center()
view3._remote_call('setSize', target='Widget', args=['500px','300px'])
view3.layout.margin = "auto"
box = ipywidgets.HBox([view1, view2, view3])
box

In [31]:
# Check & Fix PDB
# Import module
from biobb_model.model.fix_side_chain import fix_side_chain

# Create prop dict and inputs/outputs
AWH_fixed_pdb = str(AWH_dir / f'{ref}_clean.pdb')

# Create and launch bb
fix_side_chain(input_pdb_path=merged_pdb, 
               output_pdb_path=AWH_fixed_pdb)

2026-05-19 12:58:59,851 [MainThread  ] [INFO ]  Module: biobb_model.model.fix_side_chain Version: 5.2.1
2026-05-19 12:58:59,852 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_173ffde7-c0fa-4fb6-acda-3436991341bc
2026-05-19 12:58:59,853 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/haddock_best_merged.pdb --> sandbox_173ffde7-c0fa-4fb6-acda-3436991341bc
2026-05-19 12:58:59,853 [MainThread  ] [INFO ]  Launching command (it may take a while): check_structure -i /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_173ffde7-c0fa-4fb6-acda-3436991341bc/haddock_best_merged.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_173ffde7-c0fa-4fb6-acda-3436991341bc/4G6M_HL:A_clean.pdb --force_save fixside --fix ALL


2026-05-19 12:59:00,143 [MainThread  ] [INFO ]  Command 'check_structure -i /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/...' finalized with exit code 0
2026-05-19 12:59:00,144 [MainThread  ] [INFO ]  ================================================================================
=                   BioBB structure checking utility v3.15.6                   =
=            P. Andrio, A. Hospital, G. Bayarri, J.L. Gelpi 2018-23            =

Structure /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_173ffde7-c0fa-4fb6-acda-3436991341bc/haddock_best_merged.pdb loaded
 PDB id:  
 Title: mdanalysis frame 0: created by pdbwriter
 Experimental method: unknown
 Resolution (A): N.A.

 Num. models: 1
 Num. chains: 3 (A: Protein, H: Protein, L: Protein)
 Num. residues:  583
 Num. residues with ins. codes:  0
 Num. residues with H atoms: 0
 Num. HETATM residues:  0
 Num. ligands or modified residues:  0
 Num. water mol.:  0
 Num. atoms:  4501
Running 

0

In [79]:
!ln {(MD_dir / 'charmm36-jul2017.ff').resolve()} {AWH_dir / 'charmm36-jul2017.ff'} -s

In [52]:
!head {AWH_fixed_pdb}

ATOM      1  N   PRO A   2      24.553  34.539 -17.574  1.00 53.15           N
ATOM      2  CA  PRO A   2      23.173  34.761 -18.026  1.00 50.40           C
ATOM      3  C   PRO A   2      22.546  33.533 -18.688  1.00 51.75           C
ATOM      4  O   PRO A   2      23.204  32.839 -19.466  1.00 52.31           O
ATOM      5  CB  PRO A   2      23.305  35.941 -19.000  1.00 53.17           C
ATOM      6  CG  PRO A   2      24.747  35.948 -19.432  1.00 60.19           C
ATOM      7  CD  PRO A   2      25.535  34.992 -18.577  1.00 56.86           C
ATOM      8  N   VAL A   3      21.263  33.271 -18.375  1.00 45.57           N
ATOM      9  CA  VAL A   3      20.476  32.146 -18.902  1.00 43.70           C
ATOM     10  C   VAL A   3      20.461  32.161 -20.451  1.00 46.20           C


In [32]:
# Create system topology
# Import module
from biobb_gromacs.gromacs.pdb2gmx import pdb2gmx

# Create inputs/outputs
AWH_output_pdb2gmx_gro     = str(AWH_dir / f'{ref}_pdb2gmx.gro')
AWH_output_pdb2gmx_top_zip = str(AWH_dir / f'{ref}_pdb2gmx_top.zip')

prop = {
    'force_field' : 'charmm36-jul2017',
    'gmx_lib' : AWH_dir,
    'water_type' : 'tip3p',
}

# Create and launch bb
pdb2gmx(input_pdb_path=AWH_fixed_pdb, 
        output_gro_path=AWH_output_pdb2gmx_gro, 
        output_top_zip_path=AWH_output_pdb2gmx_top_zip,
        properties=prop
)

2026-05-19 12:59:03,605 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.pdb2gmx Version: 5.2.1
2026-05-19 12:59:03,605 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8
2026-05-19 12:59:03,606 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_clean.pdb --> sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8
2026-05-19 12:59:03,606 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8/4G6M_HL:A_clean.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8/4G6M_HL:A_pdb2gmx.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp


2026-05-19 12:59:03,852 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody...' finalized with exit code 0
2026-05-19 12:59:03,852 [MainThread  ] [INFO ]  Using the Charmm36-jul2017 force field in directory data/bio2/4_AWH/charmm36-jul2017.ff

going to rename data/bio2/4_AWH/charmm36-jul2017.ff/merged.r2b
Reading /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8/4G6M_HL:A_clean.pdb...
Read '', 4501 atoms

Analyzing pdb file
Splitting chemical chains based on TER records or chain id changing.

There are 3 chains and 0 blocks of water and 583 residues with 4501 atoms

  chain  #res #atoms

  1 'A'   150   1201  

  2 'H'   220   1649  

  3 'L'   213   1651  

there were 126 atoms with zero occupancy and 0 atoms with          occupancy unequal to one (out of 4501 atoms). Check your pdb file.

Reading residue database... (Charmm36-jul2017)

Preserving residues na

2026-05-19 12:59:03,853 [MainThread  ] [INFO ]                 :-) GROMACS - gmx pdb2gmx, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright pdb2gmx -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8/4G6M_HL:A_clean.pdb -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_34e44a55-37ec-4b03-8afb-0c23da1d45d8/4G6M_HL:A_pdb2gmx.gro -p p2g.top -water tip3p -ff charmm36-jul2017 -i posre.itp

Opening force field file data/bio2/4_AWH/charmm36-jul2017.ff/merged.r2b
there were 126 atoms with zero occupancy and 0 atoms with          occupancy unequal to one (out of 4501 atoms). Check your pdb file.
Opening force field file data/bio2/4_AWH/charmm36-jul2017.ff/ato

2026-05-19 12:59:03,854 [MainThread  ] [INFO ]  Compressing topology to: data/bio2/4_AWH/4G6M_HL:A_pdb2gmx_top.zip
2026-05-19 12:59:03,883 [MainThread  ] [INFO ]  Adding:
2026-05-19 12:59:03,884 [MainThread  ] [INFO ]  ['data/bio2/4_AWH/charmm36-jul2017.ff/cmap.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/ffbonded.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/ffnonbonded.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/forcefield.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/gb.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/ions.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/nbfix.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/old_c36_cmap.itp', 'data/bio2/4_AWH/charmm36-jul2017.ff/tip3p.itp', 'p2g.top', 'p2g_Protein_chain_A.itp', 'p2g_Protein_chain_H.itp', 'p2g_Protein_chain_L.itp', 'posre_Protein_chain_A.itp', 'posre_Protein_chain_H.itp', 'posre_Protein_chain_L.itp']
2026-05-19 12:59:03,884 [MainThread  ] [INFO ]  to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_

0

In [33]:
# Editconf: Create solvent box
# Import module
from biobb_gromacs.gromacs.editconf import editconf

# Create prop dict and inputs/outputs
AWH_output_editconf_gro = str(AWH_dir / f'{ref}_editconf.gro')

prop = {
    'box_type': 'dodecahedron',
    'distance_to_molecule': 0.7,
    'center_molecule': True,
}

#Create and launch bb
editconf(input_gro_path=AWH_output_pdb2gmx_gro, 
         output_gro_path=AWH_output_editconf_gro,
         properties=prop
)

2026-05-19 12:59:05,878 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.editconf Version: 5.2.1
2026-05-19 12:59:05,879 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350
2026-05-19 12:59:05,879 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_pdb2gmx.gro --> sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350
2026-05-19 12:59:05,880 [MainThread  ] [INFO ]  Distance of the box to molecule:   0.70
2026-05-19 12:59:05,880 [MainThread  ] [INFO ]  Centering molecule in the box.
2026-05-19 12:59:05,880 [MainThread  ] [INFO ]  Box type: dodecahedron
2026-05-19 12:59:05,880 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright editconf -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350/4G6M_HL:A_pdb2gmx.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/bi

2026-05-19 12:59:05,926 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright editconf -f /home/rchaves/repo/biobb/biobb_wf_antibod...' finalized with exit code 0
2026-05-19 12:59:05,926 [MainThread  ] [INFO ]  Note that major changes are planned in future for editconf, to improve usability and utility.
Read 8946 atoms
Volume: 304.981 nm^3, corresponds to roughly 137200 electrons
No velocities found
    system size :  5.273  5.147 11.237 (nm)
    diameter    : 11.278               (nm)
    center      :  0.088  2.635  0.966 (nm)
    box vectors :  5.272  5.148 11.238 (nm)
    box angles  :  90.00  90.00  90.00 (degrees)
    box volume  : 304.98               (nm^3)
    shift       :  9.420  6.874  3.516 (nm)
new center      :  9.509  9.509  4.483 (nm)
new box vectors : 12.678 12.678 12.678 (nm)
new box angles  :  60.00  60.00  90.00 (degrees)
new box volume  :1441.06               (nm^3)



2026-05-19 12:59:05,927 [MainThread  ] [INFO ]                 :-) GROMACS - gmx editconf, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright editconf -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350/4G6M_HL:A_pdb2gmx.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350/4G6M_HL:A_editconf.gro -bt dodecahedron -d 0.7 -c


GROMACS reminds you: "Computers are like humans - they do everything except think." (John von Neumann)




2026-05-19 12:59:05,927 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_43e7cd4e-0aa1-40fb-b27e-540ae33a0350']


0

In [34]:
# Solvate: Fill the box with water molecules
from biobb_gromacs.gromacs.solvate import solvate

# Create prop dict and inputs/outputs
AWH_output_solvate_gro     = str(AWH_dir / f'{ref}_solvate.gro')
AWH_output_solvate_top_zip = str(AWH_dir / f'{ref}_solvate_top.zip')

# Create and launch bb
solvate(input_solute_gro_path=AWH_output_editconf_gro, 
        output_gro_path=AWH_output_solvate_gro, 
        input_top_zip_path=AWH_output_pdb2gmx_top_zip, 
        output_top_zip_path=AWH_output_solvate_top_zip,
)

2026-05-19 12:59:09,129 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.solvate Version: 5.2.1
2026-05-19 12:59:09,130 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_49cf25bf-360f-4ea6-88fc-955e05aa213f
2026-05-19 12:59:09,130 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_editconf.gro --> sandbox_49cf25bf-360f-4ea6-88fc-955e05aa213f
2026-05-19 12:59:09,136 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_pdb2gmx_top.zip
2026-05-19 12:59:09,136 [MainThread  ] [INFO ]  to:
2026-05-19 12:59:09,136 [MainThread  ] [INFO ]  ['5f166248-f3be-4032-81f7-fd87005aaa87/cmap.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/ffbonded.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/ffnonbonded.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/forcefield.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/gb.itp', '5f166248-f3be-403

2026-05-19 12:59:09,485 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright solvate -cp /home/rchaves/repo/biobb/biobb_wf_antibod...' finalized with exit code 0
2026-05-19 12:59:09,486 [MainThread  ] [INFO ]  
         based on residue and atom names, since they could not be
         definitively assigned from the information in your input
         files. These guessed numbers might deviate from the mass
         and radius of the atom type. Please check the output
         files if necessary. Note, that this functionality may
         be removed in a future GROMACS version. Please, consider
         using another file format for your input.

         The atomic radii are set according to:

++++ PLEASE READ AND CITE THE FOLLOWING REFERENCE ++++
A. Bondi
van der Waals Volumes and Radii
J. Phys. Chem. (1964)
https://doi.org/10.1021/j100785a001
-------- -------- --- Thank You --- -------- --------

Adding line for 44428 solvent molecules with resname (SOL) to topology file (5f1662

2026-05-19 12:59:09,486 [MainThread  ] [INFO ]                 :-) GROMACS - gmx solvate, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright solvate -cp /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_49cf25bf-360f-4ea6-88fc-955e05aa213f/4G6M_HL:A_editconf.gro -cs spc216.gro -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_49cf25bf-360f-4ea6-88fc-955e05aa213f/4G6M_HL:A_solvate.gro -p 5f166248-f3be-4032-81f7-fd87005aaa87/p2g.top

Reading solute configuration
Reading solvent configuration

Initialising inter-atomic distances...
Generating solvent configuration
Will generate new solvent configuration of 7x7x5 boxes
Solvent box contains 157269 atoms in 52423 residues
Removed 15738 solvent 

2026-05-19 12:59:09,492 [MainThread  ] [INFO ]  Compressing topology to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_49cf25bf-360f-4ea6-88fc-955e05aa213f/4G6M_HL:A_solvate_top.zip
2026-05-19 12:59:09,493 [MainThread  ] [INFO ]  Ignored file 5f166248-f3be-4032-81f7-fd87005aaa87/data/bio2/4_AWH/charmm36-jul2017.ff/forcefield.itp
2026-05-19 12:59:09,503 [MainThread  ] [INFO ]  Ignored file 5f166248-f3be-4032-81f7-fd87005aaa87/data/bio2/4_AWH/charmm36-jul2017.ff/tip3p.itp
2026-05-19 12:59:09,503 [MainThread  ] [INFO ]  Ignored file 5f166248-f3be-4032-81f7-fd87005aaa87/data/bio2/4_AWH/charmm36-jul2017.ff/ions.itp
2026-05-19 12:59:09,509 [MainThread  ] [INFO ]  Adding:
2026-05-19 12:59:09,510 [MainThread  ] [INFO ]  ['5f166248-f3be-4032-81f7-fd87005aaa87/p2g.top', '5f166248-f3be-4032-81f7-fd87005aaa87/p2g_Protein_chain_A.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/p2g_Protein_chain_H.itp', '5f166248-f3be-4032-81f7-fd87005aaa87/p2g_Protein_chain_L.itp', '5f1

0

In [35]:
# Show protein
view = nv.show_structure_file(str(AWH_output_solvate_gro))
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute')
view.add_representation(repr_type='ball+stick', selection='SOL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

In [36]:
# Grompp: Creating portable binary run file for ion generation
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
AWH_output_gppion_tpr = str(AWH_dir / f'{ref}_gppion.tpr')
prop = {
    'simulation_type': 'ions',
    'gmx_lib' : Path('.').resolve(),
    'maxwarn': 2
}

# Create and launch bb
grompp(input_gro_path=AWH_output_solvate_gro, 
       input_top_zip_path=AWH_output_solvate_top_zip, 
       output_tpr_path=AWH_output_gppion_tpr,
       properties=prop
)

2026-05-19 12:59:30,980 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-19 12:59:30,980 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3
2026-05-19 12:59:30,981 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_solvate.gro --> sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3
2026-05-19 12:59:30,985 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_solvate_top.zip
2026-05-19 12:59:30,985 [MainThread  ] [INFO ]  to:
2026-05-19 12:59:30,985 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/p2g.top', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/p2g_Protein_chain_A.itp', '/home/rch

2026-05-19 12:59:36,948 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/...' finalized with exit code 0
2026-05-19 12:59:36,948 [MainThread  ] [INFO ]  Setting the LD random seed to -201367874

Generated 97877 of the 97903 non-bonded parameter combinations

Generated 64492 of the 97903 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

Excluding 3 bonded neighbours molecule type 'Protein_chain_H'

Excluding 3 bonded neighbours molecule type 'Protein_chain_L'

Excluding 2 bonded neighbours molecule type 'SOL'

++++ PLEASE READ AND CITE THE FOLLOWING REFERENCE ++++
J. S. Hub, B. L. de Groot, H. Grubmüller, G. Groenhof
Quantifying Artifacts in Ewald Simulations of Inhomogeneous Systems with a Net
Charge
J. Chem. Theory Comput. (2014)
https://doi.org/10.1021/ct400626b
-------- -------- --- Thank You --- -------- --------

Analysing residue names:
There are:   583    Protein residues
The

2026-05-19 12:59:36,949 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/4G6M_HL:A_solvate.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/4G6M_HL:A_solvate.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/noteboo

2026-05-19 12:59:36,955 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7aeda32b-634f-430f-ad8f-eb132fed07c3']
2026-05-19 12:59:36,956 [MainThread  ] [INFO ]  


0

In [37]:
# Show protein
view = nv.show_structure_file(AWH_output_solvate_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='sstruc')
view.add_representation(repr_type='ball+stick', selection='NA')
view.add_representation(repr_type='ball+stick', selection='CL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

In [38]:
# Genion: Adding ions to neutralize the system
from biobb_gromacs.gromacs.genion import genion

# Create prop dict and inputs/outputs
AWH_output_genion_gro     = str(AWH_dir / f'{ref}_genion.gro')
AWH_output_genion_top_zip = str(AWH_dir / f'{ref}_genion_top.zip')
prop={
    'neutral':True,
    'ionic_concentration' : 0.15
}

# Create and launch bb
genion(input_tpr_path=AWH_output_gppion_tpr, 
       output_gro_path=AWH_output_genion_gro, 
       input_top_zip_path=AWH_output_solvate_top_zip, 
       output_top_zip_path=AWH_output_genion_top_zip, 
       properties=prop
)

2026-05-19 12:59:44,615 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.genion Version: 5.2.1
2026-05-19 12:59:44,616 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00269669-7a29-484c-898c-48fa1c29fa2e
2026-05-19 12:59:44,616 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_gppion.tpr --> sandbox_00269669-7a29-484c-898c-48fa1c29fa2e
2026-05-19 12:59:44,618 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/accdf3b1-16e4-43bf-a87a-0490f46147dd.stdin --> sandbox_00269669-7a29-484c-898c-48fa1c29fa2e
2026-05-19 12:59:44,620 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_solvate_top.zip
2026-05-19 12:59:44,621 [MainThread  ] [INFO ]  to:
2026-05-19 12:59:44,621 [MainThread  ] [INFO ]  ['c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g.top', 'c1c2d7

2026-05-19 12:59:44,731 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright genion -s /home/rchaves/repo/biobb/biobb_wf_antibody/...' finalized with exit code 0
2026-05-19 12:59:44,731 [MainThread  ] [INFO ]  Will try to add 0 NA ions and 6 CL ions.
Select a continuous group of solvent molecules
Selected 13: 'SOL'

Processing topology
Replacing 6 solute molecules in topology file (c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g.top)  by 0 NA and 6 CL ions.



/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/biobb_common/generic/biobb_object.py:187: UserWarning: Warning: ionic_concentration is not a recognized property. The most similar property is: concentration
  warnings.warn(
2026-05-19 12:59:44,732 [MainThread  ] [INFO ]                  :-) GROMACS - gmx genion, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright genion -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00269669-7a29-484c-898c-48fa1c29fa2e/4G6M_HL:A_gppion.tpr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00269669-7a29-484c-898c-48fa1c29fa2e/4G6M_HL:A_genion.gro -p c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g.top -neutral -seed 1993

Reading f

2026-05-19 12:59:44,736 [MainThread  ] [INFO ]  Compressing topology to: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_00269669-7a29-484c-898c-48fa1c29fa2e/4G6M_HL:A_genion_top.zip
2026-05-19 12:59:44,737 [MainThread  ] [INFO ]  Ignored file c1c2d745-3673-4a16-9d10-ce5afa98936e/data/bio2/4_AWH/charmm36-jul2017.ff/forcefield.itp
2026-05-19 12:59:44,747 [MainThread  ] [INFO ]  Ignored file c1c2d745-3673-4a16-9d10-ce5afa98936e/data/bio2/4_AWH/charmm36-jul2017.ff/tip3p.itp
2026-05-19 12:59:44,748 [MainThread  ] [INFO ]  Ignored file c1c2d745-3673-4a16-9d10-ce5afa98936e/data/bio2/4_AWH/charmm36-jul2017.ff/ions.itp
2026-05-19 12:59:44,752 [MainThread  ] [INFO ]  Adding:
2026-05-19 12:59:44,753 [MainThread  ] [INFO ]  ['c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g.top', 'c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g_Protein_chain_A.itp', 'c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g_Protein_chain_H.itp', 'c1c2d745-3673-4a16-9d10-ce5afa98936e/p2g_Protein_chain_L.itp', 'c1c2

0

In [39]:
# Show protein
view = nv.show_structure_file(AWH_output_genion_gro)
view.clear_representations()
view.add_representation(repr_type='cartoon', selection='solute', color='sstruc')
view.add_representation(repr_type='ball+stick', selection='NA')
view.add_representation(repr_type='ball+stick', selection='CL')
view._remote_call('setSize', target='Widget', args=['','600px'])
view.camera='orthographic'
view

NGLWidget()

### Minimization and equilibration

#### Minimization

In [40]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
AWH_output_gppmin_tpr = AWH_dir / f'{ref}_gppmin.tpr'

prop = {
    'gmx_lib' : Path('.').resolve(),
    'maxwarn' : 1
}

# Create and launch bb
grompp(input_gro_path=AWH_output_genion_gro, 
       input_top_zip_path=AWH_output_genion_top_zip, 
       input_mdp_path=input_mdp_min,
       output_tpr_path=AWH_output_gppmin_tpr,  
       properties=prop
)

# Mdrun: Running NVT minimization
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
AWH_output_min_trr = AWH_dir / f'{ref}_min.trr'
AWH_output_min_gro = AWH_dir / f'{ref}_min.gro'
AWH_output_min_edr = AWH_dir / f'{ref}_min.edr'
AWH_output_min_log = AWH_dir / f'{ref}_min.log'

# Create and launch bb
mdrun(input_tpr_path=AWH_output_gppmin_tpr, 
      output_trr_path=AWH_output_min_trr, 
      output_gro_path=AWH_output_min_gro, 
      output_edr_path=AWH_output_min_edr, 
      output_log_path=AWH_output_min_log,
)

2026-05-19 12:59:52,446 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-19 12:59:52,447 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0
2026-05-19 12:59:52,448 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_genion.gro --> sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0
2026-05-19 12:59:52,450 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/emin-charmm.mdp --> sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0
2026-05-19 12:59:52,453 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_genion_top.zip
2026-05-19 12:59:52,454 [MainThread  ] [INFO ]  to:
2026-05-19 12:59:52,454 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0/p2g.top', '/home/rchav

2026-05-19 12:59:58,312 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0/4G6M_HL:A_genion.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0/4G6M_HL:A_genion.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks

2026-05-19 12:59:58,319 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_24e2be37-2e47-41ce-a81e-bd98506d8cd0']
2026-05-19 12:59:58,320 [MainThread  ] [INFO ]  
2026-05-19 12:59:58,338 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-19 12:59:58,339 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb
2026-05-19 12:59:58,340 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_gppmin.tpr --> sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb
2026-05-19 12:59:58,341 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb/4G6M_HL:A_min.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/noteboo

2026-05-19 13:01:28,450 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb/4G6M_HL:A_min.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb/4G6M_HL:A_gppmin.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb/4G6M_HL:A_min.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb/4G6M_HL:A_min.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibo

2026-05-19 13:01:28,458 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c0ee1253-eb21-41c7-9ef1-b3d461c399fb']
2026-05-19 13:01:28,458 [MainThread  ] [INFO ]  


0

In [41]:
# GMXEnergy: Getting system energy by time  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
AWH_output_min_ene_xvg = AWH_dir / f'{ref}_min_ene.xvg'
prop = {
    'terms':  ["Potential"]
}

# Create and launch bb
gmx_energy(input_energy_path=AWH_output_min_edr, 
          output_xvg_path=AWH_output_min_ene_xvg, 
          properties=prop
)

import plotly
import plotly.graph_objs as go

#Read data from file and filter energy values higher than 1000 Kj/mol^-1
with open(AWH_output_min_ene_xvg,'r') as energy_file:
    x,y = map(
        list,
        zip(*[
            (float(line.split()[0]),float(line.split()[1]))
            for line in energy_file 
            if not line.startswith(("#","@")) 
            if float(line.split()[1]) < 1000 
        ])
    )

plotly.offline.init_notebook_mode(connected=True)

fig = {
    "data": [go.Scatter(x=x, y=y)],
    "layout": go.Layout(title="Energy Minimization",
                        xaxis=dict(title = "Energy Minimization Step"),
                        yaxis=dict(title = "Potential Energy KJ/mol-1")
                       )
}

plotly.offline.iplot(fig)

2026-05-19 13:01:28,690 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-05-19 13:01:28,691 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12
2026-05-19 13:01:28,691 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_min.edr --> sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12
2026-05-19 13:01:28,692 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12/4G6M_HL:A_min.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12/4G6M_HL:A_min_ene.xvg -xvg none < /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427

2026-05-19 13:01:28,708 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12/4G6M_HL:A_min.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12/4G6M_HL:A_min_ene.xvg -xvg none

Opened /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12/4G6M_HL:A_min.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.


2026-05-19 13:01:28,709 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_0718c518-e427-4ed3-be00-b72ccaa4fa12']
2026-05-19 13:01:28,709 [MainThread  ] [INFO ]  


#### NVT equilibration

In [42]:
# Grompp: Creating portable binary run file for NVT Equilibration
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
AWH_output_nvt_tpr = AWH_dir / f'{ref}_nvt.tpr'
prop = {
    'mdp':{
        'nsteps': 5000,
        'dt': 0.002,
        'Define': '-DPOSRES',
    },
    'simulation_type': 'nvt',
    'gmx_lib' : Path('.').resolve(),
}

# Create and launch bb
grompp(input_gro_path=AWH_output_min_gro, 
       input_top_zip_path=AWH_output_genion_top_zip, 
       output_tpr_path=AWH_output_nvt_tpr,  
       properties=prop)

# Mdrun: Running Equilibration NVT
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
AWH_output_nvt_trr = AWH_dir / f'{ref}_nvt.trr'
AWH_output_nvt_gro = AWH_dir / f'{ref}_nvt.gro'
AWH_output_nvt_edr = AWH_dir / f'{ref}_nvt.edr'
AWH_output_nvt_log = AWH_dir / f'{ref}_nvt.log'
AWH_output_nvt_cpt = AWH_dir / f'{ref}_nvt.cpt'

# Create and launch bb
mdrun(input_tpr_path=AWH_output_nvt_tpr, 
      output_trr_path=AWH_output_nvt_trr, 
      output_gro_path=AWH_output_nvt_gro, 
      output_edr_path=AWH_output_nvt_edr, 
      output_log_path=AWH_output_nvt_log, 
      output_cpt_path=AWH_output_nvt_cpt)

2026-05-19 13:01:29,085 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-19 13:01:29,085 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d
2026-05-19 13:01:29,086 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_min.gro --> sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d
2026-05-19 13:01:29,091 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_genion_top.zip
2026-05-19 13:01:29,091 [MainThread  ] [INFO ]  to:
2026-05-19 13:01:29,092 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/p2g.top', '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/p2g_Protein_chain_A.itp', '/home/rchaves/

2026-05-19 13:01:34,983 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/...' finalized with exit code 0
2026-05-19 13:01:34,983 [MainThread  ] [INFO ]  Setting the LD random seed to -705708035

Generated 97877 of the 97903 non-bonded parameter combinations

Generated 64492 of the 97903 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'Protein_chain_H'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'Protein_chain_L'

turning H bonds into constraints...

Excluding 2 bonded neighbours molecule type 'SOL'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'CL'

turning H bonds into constraints...

Setting gen_seed to -1425551717

Velocities were taken from a Maxwell distribution at 300 K
Analysing residue names:
There are:   583    Protein 

2026-05-19 13:01:34,984 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/4G6M_HL:A_min.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/4G6M_HL:A_min.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandb

2026-05-19 13:01:34,987 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_c6ed31a7-0a61-4cf7-a107-fa113ff7bc2d']
2026-05-19 13:01:34,987 [MainThread  ] [INFO ]  
2026-05-19 13:01:35,005 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-19 13:01:35,006 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00
2026-05-19 13:01:35,006 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_nvt.tpr --> sandbox_76413a1b-ab81-40a8-8645-62d6129cac00
2026-05-19 13:01:35,009 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00/4G6M_HL:A_nvt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/

2026-05-19 13:03:26,091 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00/4G6M_HL:A_nvt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00/4G6M_HL:A_nvt.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00/4G6M_HL:A_nvt.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00/4G6M_HL:A_nvt.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/

2026-05-19 13:03:26,134 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_76413a1b-ab81-40a8-8645-62d6129cac00', 'traj_comp.xtc']
2026-05-19 13:03:26,135 [MainThread  ] [INFO ]  


0

In [43]:
# GMXEnergy: Getting system temperature by time during NVT Equilibration  
from biobb_analysis.gromacs.gmx_energy import gmx_energy

# Create prop dict and inputs/outputs
AWH_output_nvt_temp_xvg = AWH_dir / f'{ref}_nvt_temp.xvg'
prop = {
    'terms':  ["Temperature"]
}

# Create and launch bb
gmx_energy(input_energy_path=AWH_output_nvt_edr, 
          output_xvg_path=AWH_output_nvt_temp_xvg, 
          properties=prop)

# Read temperature data from file
with open(AWH_output_nvt_temp_xvg, 'r') as temperature_file:
    x, y = zip(*[
        (float(line.split()[0]), float(line.split()[1]))
        for line in temperature_file
        if not line.startswith(("#", "@"))
    ])

# Create a scatter plot
fig = go.Figure(data=go.Scatter(x=x, y=y, mode='lines+markers'))

# Update layout
fig.update_layout(title="Temperature during NVT Equilibration",
                  xaxis_title="Time (ps)",
                  yaxis_title="Temperature (K)",
                  height=600)
fig.show()

2026-05-19 13:03:26,157 [MainThread  ] [INFO ]  Module: biobb_analysis.gromacs.gmx_energy Version: 5.2.1
2026-05-19 13:03:26,158 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30
2026-05-19 13:03:26,158 [MainThread  ] [INFO ]  Copy to stage: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_nvt.edr --> sandbox_710eb446-b152-46c6-bfd9-6e761f316d30
2026-05-19 13:03:26,159 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30/4G6M_HL:A_nvt.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30/4G6M_HL:A_nvt_temp.xvg -xvg none < /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b15

2026-05-19 13:03:26,171 [MainThread  ] [INFO ]                  :-) GROMACS - gmx energy, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx energy -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30/4G6M_HL:A_nvt.edr -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30/4G6M_HL:A_nvt_temp.xvg -xvg none

Opened /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30/4G6M_HL:A_nvt.edr as single precision energy file

Select the terms you want from the following list by
selecting either (part of) the name or the number or a combination.
End your selection with an empty line or a zero.

2026-05-19 13:03:26,171 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_710eb446-b152-46c6-bfd9-6e761f316d30']
2026-05-19 13:03:26,171 [MainThread  ] [INFO ]  


#### NPT Equilibration

In [44]:
# Grompp: Creating portable binary run file for NPT Equilibration
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs
AWH_output_npt_tpr = AWH_dir / f'{ref}_npt.tpr'

prop = {
    'mdp': {
        'nsteps' : 5000, # subir
    },
    'gmx_lib' : Path('.').resolve(),
    'maxwarn' : 1,
}

# Create and launch bb
grompp(input_gro_path=AWH_output_nvt_gro, 
       input_top_zip_path=AWH_output_genion_top_zip, 
       input_mdp_path=input_mdp_eq,
       output_tpr_path=AWH_output_npt_tpr,  
       properties=prop
)

2026-05-19 13:03:26,648 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.grompp Version: 5.2.1
2026-05-19 13:03:26,649 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721
2026-05-19 13:03:26,649 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_nvt.gro --> sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721
2026-05-19 13:03:26,652 [MainThread  ] [INFO ]  Copy to stage: data/bio2/3_MD/md_eq_posre_charmm36m.mdp --> sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721
2026-05-19 13:03:26,656 [MainThread  ] [INFO ]  Extracting: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/data/bio2/4_AWH/4G6M_HL:A_genion_top.zip
2026-05-19 13:03:26,656 [MainThread  ] [INFO ]  to:
2026-05-19 13:03:26,657 [MainThread  ] [INFO ]  ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/p2g.top', '/hom

2026-05-19 13:03:32,653 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/...' finalized with exit code 0
2026-05-19 13:03:32,653 [MainThread  ] [INFO ]  Setting the LD random seed to -940574993

Generated 97877 of the 97903 non-bonded parameter combinations

Generated 64492 of the 97903 1-4 parameter combinations

Excluding 3 bonded neighbours molecule type 'Protein_chain_A'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'Protein_chain_H'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'Protein_chain_L'

turning H bonds into constraints...

Excluding 2 bonded neighbours molecule type 'SOL'

turning H bonds into constraints...

Excluding 1 bonded neighbours molecule type 'CL'

turning H bonds into constraints...

Taking velocities from '/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/4G6M_

2026-05-19 13:03:32,653 [MainThread  ] [INFO ]                  :-) GROMACS - gmx grompp, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright grompp -f /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/grompp.mdp -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/4G6M_HL:A_nvt.gro -r /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/4G6M_HL:A_nvt.gro -p /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721/p2g.top -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandb

2026-05-19 13:03:32,670 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_9db4655b-0907-4a25-8c59-e3b5e6fef721']
2026-05-19 13:03:32,670 [MainThread  ] [INFO ]  


0

In [45]:
# Mdrun: Running NPT System Equilibration
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
AWH_output_npt_trr = AWH_dir / f'{ref}_npt.trr'
AWH_output_npt_gro = AWH_dir / f'{ref}_npt.gro'
AWH_output_npt_edr = AWH_dir / f'{ref}_npt.edr'
AWH_output_npt_log = AWH_dir / f'{ref}_npt.log'
AWH_output_npt_cpt = AWH_dir / f'{ref}_npt.cpt'

# Create and launch bb
mdrun(input_tpr_path=AWH_output_npt_tpr, 
      output_trr_path=AWH_output_npt_trr, 
      output_gro_path=AWH_output_npt_gro, 
      output_edr_path=AWH_output_npt_edr, 
      output_log_path=AWH_output_npt_log, 
      output_cpt_path=AWH_output_npt_cpt,
)

2026-05-19 13:03:32,878 [MainThread  ] [INFO ]  Module: biobb_gromacs.gromacs.mdrun Version: 5.2.1
2026-05-19 13:03:32,879 [MainThread  ] [INFO ]  Directory successfully created: /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4
2026-05-19 13:03:32,879 [MainThread  ] [INFO ]  Copy to stage: data/bio2/4_AWH/4G6M_HL:A_npt.tpr --> sandbox_7490555c-feb3-422c-aa6d-f714737528d4
2026-05-19 13:03:32,882 [MainThread  ] [INFO ]  Launching command (it may take a while): gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.gro -e /home/rchaves/repo/biob

2026-05-19 13:05:33,610 [MainThread  ] [INFO ]  Command 'gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/b...' finalized with exit code 0


2026-05-19 13:05:33,610 [MainThread  ] [INFO ]                  :-) GROMACS - gmx mdrun, 2026.0-conda_forge (-:

Executable:   /home/rchaves/miniforge3/envs/biobb_wf_antibody/bin.AVX2_256/gmx
Data prefix:  /home/rchaves/miniforge3/envs/biobb_wf_antibody
Working dir:  /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks
Command line:
  gmx -nobackup -nocopyright mdrun -o /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.trr -s /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.tpr -c /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.gro -e /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4/4G6M_HL:A_npt.edr -g /home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/

2026-05-19 13:05:33,625 [MainThread  ] [INFO ]  Removed: ['/home/rchaves/repo/biobb/biobb_wf_antibody/biobb_wf_antibody/notebooks/sandbox_7490555c-feb3-422c-aa6d-f714737528d4', 'traj_comp.xtc']
2026-05-19 13:05:33,625 [MainThread  ] [INFO ]  Path data/bio2/4_AWH/4G6M_HL:A_npt.trr --- biobb_gromacs.gromacs.mdrun: Unexisting output_trr_path file.
2026-05-19 13:05:33,626 [MainThread  ] [INFO ]  


/home/rchaves/miniforge3/envs/biobb_wf_antibody/lib/python3.12/site-packages/biobb_common/tools/file_utils.py:801: UserWarning: Path data/bio2/4_AWH/4G6M_HL:A_npt.trr --- biobb_gromacs.gromacs.mdrun: Unexisting output_trr_path file.
  warnings.warn(not_found_error_string)


0

### AWH

In [15]:
import MDAnalysis as mda
u_gro = mda.Universe(AWH_output_npt_gro).select_atoms('protein')
u_fix = mda.Universe(fixed_pdb).select_atoms('protein')  

In [17]:
u_gro.n_atoms, u_gro.n_residues, u_fix.n_atoms, u_fix.n_residues

(8946, 583, 3291, 432)

In [64]:
# Calculate the distance between paratope and epitope residues
paratope_residues = u.select_atoms(f"resid {' '.join(map(str, paratope))} and chainID A").residues
epitope_residues = u.select_atoms(f"resid {' '.join(map(str, epitope))} and chainID B").residues
distances = mda.lib.distances.distance_array(paratope_residues.atoms.positions, epitope_residues.atoms.positions)
# Get the average distancebetween paratope and epitope residues
average_distance = distances.mean()
print(f"Average distance between paratope and epitope residues: {average_distance:.2f} Å")

Average distance between paratope and epitope residues: nan Å


/tmp/ipykernel_7076/43612369.py:6: RuntimeWarning: Mean of empty slice
  average_distance = distances.mean()


In [ ]:
from biobb_gromacs.gromacs.make_ndx import make_ndx

# Create prop dict and inputs/outputs
loop_ndx = MD_dir / f'{ab}_loop.ndx'

prop = { 
    'selection': 'r 27-38 | r 56-65 | r 105-117\nname 10 Loop'
}

# Create and launch bb
make_ndx(input_structure_path=anarcii_gro,
         output_ndx_path=loop_ndx,
         properties=prop)

In [ ]:
# https://github.com/alevil-gmx/workflow_template/blob/main/workflow_template/data/input/md_charmm36m.mdp
# ns_type -> ns-type
input_mdp_md = AWH_dir / "awh_md_charmm36m.mdp"

file=""";define                   =  -DPOSRES
;include                  = -I/home/villa/work/UseCaseI/forcefield
integrator               = md
tinit                    = 0
dt                       = 0.002
nsteps                   = 50000000   ; 100  ns
;nstcomm                  = 1
comm_grps                = system
nstxout                  = 0
nstvout                  = 0
nstfout                  = 0
nstlog                   = 100000
nstenergy                = 100000
nstxout-compressed       = 100000
compressed-x-grps        = system 
nstlist                  = 10
ns-type                  = grid
pbc                      = xyz
; neighbour
; rlist                   = 1.2 ; not used when cutoff-scheme = verlet
cutoff-scheme           = Verlet
; coulomb
coulombtype              = PME
rcoulomb                 = 1.2
fourierspacing           = 0.15 ;  For constant accuracy one should keep fourier-spacing/rcoulomb constant = 0.125
; vdw
vdwtype                  = Cut-off
vdw_modifier             = Force-switch
rvdw_switch              = 1.0  ; 
rvdw                     = 1.2
DispCorr                 = EnerPres ; while for lipid bilayer,  it's a difficult topic in the CHARMM force field. If one don't have lipids bi- or monolayers one should use it. 
;
constraints              = h-bonds
constraint_algorithm     = LINCS
; barostat
Pcoupl                   = Parrinello-Rahman   
tau_p                    = 5.0  
ref_p                    = 1.0 
compressibility          = 4.5e-5 
; thermostat
Tcoupl                   = v-rescale 
tc-grps                  = system
tau_t                    = 0.5 ; or 0.1 
ref_t                    = 298 """
with open(input_mdp_md, 'w') as f:
    f.write(file)

In [ ]:
# Grompp: Creating portable binary run file for mdrun
from biobb_gromacs.gromacs.grompp import grompp

# Create prop dict and inputs/outputs

AWH_output_gppmd_tpr = AWH_dir / f"{ref}_gppmd.tpr"

prop = {
    'mdp': {
        'nsteps' :  500_000, # 1 ns
        'nstxout' :  10_000
    },
    'gmx_lib' : AWH_dir,
    'maxwarn' : 1
}

# Create and launch bb
grompp(input_gro_path=AWH_output_npt_gro, 
       input_top_zip_path=AWH_output_genion_top_zip, 
       input_mdp_path=input_mdp_md,
       output_tpr_path=AWH_output_gppmd_tpr, 
       input_cpt_path=AWH_output_npt_cpt, 
       properties=prop
)

In [ ]:
# Mdrun: Running free dynamics
from biobb_gromacs.gromacs.mdrun import mdrun

# Create prop dict and inputs/outputs
AWH_output_md_trr = AWH_dir / f'{ref}_md.trr'
AWH_output_md_gro = AWH_dir / f'{ref}_md.gro'
AWH_output_md_edr = AWH_dir / f'{ref}_md.edr'
AWH_output_md_log = AWH_dir / f'{ref}_md.log'
AWH_output_md_cpt = AWH_dir / f'{ref}_md.cpt'

# Create and launch bb
mdrun(input_tpr_path=AWH_output_gppmd_tpr, 
      output_trr_path=AWH_output_md_trr, 
      output_gro_path=AWH_output_md_gro, 
      output_edr_path=AWH_output_md_edr, 
      output_log_path=AWH_output_md_log, 
      output_cpt_path=AWH_output_md_cpt,
)

In [ ]:
# GMXImage: "Imaging" the resulting trajectory
#           Removing water molecules and ions from the resulting structure
from biobb_analysis.gromacs.gmx_image import gmx_image

# Create prop dict and inputs/outputs
AWH_output_imaged_traj = AWH_dir / f'{ref}_imaged_traj.trr'
prop = {
    'center_selection':  'Protein',
    'output_selection': 'Protein',
    'pbc' : 'nojump',
}

# Create and launch bb
gmx_image(input_traj_path=AWH_output_md_trr,
          input_top_path=AWH_output_gppmin_tpr,
          output_traj_path=AWH_output_imaged_traj, 
          properties=prop
)

In [ ]:
# GMXTrjConvStr: Converting and/or manipulating a structure
#                Removing water molecules and ions from the resulting structure
#                The "dry" structure will be used as a topology to visualize 
#                the "imaged dry" trajectory generated in the previous step.
from biobb_analysis.gromacs.gmx_trjconv_str import gmx_trjconv_str

# Create prop dict and inputs/outputs
AWH_output_dry_gro = AWH_dir / f'{ref}_md_dry.gro'
prop = {
    'selection':  'Protein'
}

# Create and launch bb
gmx_trjconv_str(input_structure_path=AWH_output_md_gro,
                input_top_path=AWH_output_gppmd_tpr,
                output_str_path=AWH_output_dry_gro, 
                properties=prop
)

In [ ]:
# GMXImage: "Imaging" the resulting trajectory
#           Removing water molecules and ions from the resulting structure
from biobb_analysis.gromacs.gmx_image import gmx_image

# Create prop dict and inputs/outputs
AWH_output_imaged_traj_rot = AWH_dir / f'{ref}_imaged_traj_rot.trr'
prop = {
    'fit' : 'rot+trans',
    'center': False
}

# Create and launch bb
gmx_image(input_traj_path=AWH_output_imaged_traj,
          input_top_path=AWH_output_dry_gro,
          output_traj_path=AWH_output_imaged_traj_rot, 
          properties=prop
)

## PMX

In [ ]:
# https://mmb.irbbarcelona.org/biobb/workflows/tutorials/biobb_wf_pmx_tutorial
# https://github.com/bioexcel/biobb_wf_pmx_tutorial

# Use 
https://www.bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/#bonus-1-dissecting-the-interface-energetics-what-is-the-impact-of-a-single-mutation

https://www.frontiersin.org/journals/molecular-biosciences/articles/10.3389/fmolb.2023.1063971/full

https://github.com/CSB-KaracaLab/prot-on

***
<a id="questions"></a>

## Questions & Comments

Questions, issues, suggestions and comments are really welcome!

* GitHub issues:
    * [https://github.com/bioexcel/biobb](https://github.com/bioexcel/biobb)

* BioExcel forum:
    * [https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library](https://ask.bioexcel.eu/c/BioExcel-Building-Blocks-library)
